# 模块概述

WtUftCore 是 WonderTrader UFT（Ultra Fast Trading，极速交易）策略运行的核心模块，负责提供UFT策略的实时交易环境。主要包括：
- UFT策略引擎和策略上下文管理
- 实时市场数据处理和分发
- 交易适配和订单管理
- 行情解析器适配
- 策略参数共享管理
- 事件通知机制
- 数据结构和辅助工具类

1. **数据定义层**（UftDataDefs）：
   - 定义UFT策略使用的核心数据结构
   - 包括持仓明细、订单、成交、回合等数据结构
   - 采用块头+数据数组的结构，支持内存映射文件存储
   - 是整个模块的数据基础

2. **引擎层**（WtUftEngine + WtUftTicker）：
   - WtUftEngine：UFT引擎核心，管理策略上下文、数据订阅、时间管理等
   - WtUftTicker：实时ticker，处理实时行情并触发分钟线闭合事件
   - 是整个框架的控制中枢

3. **策略层**（UftStrategyMgr + UftStraContext）：
   - UftStrategyMgr：策略管理器，动态加载策略工厂，创建策略实例
   - UftStraContext：策略上下文，管理策略的交易上下文、持仓、订单、数据订阅等
   - 使用UftDataDefs定义的数据结构存储持仓、订单、成交等数据
   - 提供策略运行环境和交易接口

4. **数据层**（WtUftDtMgr）：
   - 管理实时tick、历史tick、K线等市场数据
   - 实现IDataManager接口，提供统一的数据查询接口
   - 支持数据订阅和数据切片查询

5. **适配器层**（TraderAdapter + ParserAdapter）：
   - TraderAdapter：交易适配器，适配不同的交易接口，提供统一的交易操作
   - ParserAdapter：行情解析器适配器，适配不同的行情数据源，统一行情接口

6. **工具支持层**（EventNotifier + ActionPolicyMgr + ShareManager + WtHelper）：
   - EventNotifier：事件通知器，通过消息队列异步广播交易事件
   - ActionPolicyMgr：动作策略管理器，管理交易动作的执行规则
   - ShareManager：共享内存管理器，管理策略参数的共享和持久化
   - WtHelper：辅助工具类，提供路径管理和时间管理功能


# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef engineClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef contextClass fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#000;
    classDef adapterClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef dataClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;
    classDef utilClass fill:#fce4ec,stroke:#880e4f,stroke-width:2px,color:#000;
    classDef tickerClass fill:#e0f2f1,stroke:#004d40,stroke-width:2px,color:#000;
    classDef mgrClass fill:#ffe0b2,stroke:#e65100,stroke-width:2px,color:#000;
    classDef interfaceClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef defClass fill:#fffde7,stroke:#f57f17,stroke-width:2px,color:#000;

    %% 接口层
    subgraph Interfaces["接口层 - 抽象接口"]
        direction TB
        IUftStraCtx["IUftStraCtx<br/>UFT策略上下文接口<br/>• 交易接口<br/>• 数据查询接口<br/>• 持仓管理接口"]
        ITrdNotifySink["ITrdNotifySink<br/>交易通知接口<br/>• 成交回报<br/>• 订单回报<br/>• 持仓更新<br/>• 通道状态"]
        IParserStub["IParserStub<br/>行情解析器存根接口<br/>• Tick推送<br/>• 订单队列推送<br/>• 订单明细推送<br/>• 成交明细推送"]
        IDataManager["IDataManager<br/>数据管理器接口<br/>• Tick切片查询<br/>• K线切片查询<br/>• Level-2数据查询"]
        ITraderSpi["ITraderSpi<br/>交易接口回调<br/>• 连接回调<br/>• 登录回调<br/>• 订单回调<br/>• 成交回调"]
        IParserSpi["IParserSpi<br/>行情解析器回调<br/>• Tick回调<br/>• 订单队列回调<br/>• 订单明细回调<br/>• 成交明细回调"]
    end

    %% 数据定义层
    subgraph DataDefs["数据定义层 - 数据结构"]
        direction TB
        UftDataDefs["UftDataDefs<br/>UFT数据定义<br/>• BlockHeader 数据块头<br/>• DetailStruct 持仓明细<br/>• PositionBlock 持仓块<br/>• OrderStruct 订单结构<br/>• OrderBlock 订单块<br/>• TradeStruct 成交结构<br/>• TradeBlock 成交块<br/>• RoundStruct 回合结构<br/>• RoundBlock 回合块"]:::defClass
    end

    %% 引擎层
    subgraph Engines["引擎层 - 核心控制"]
        direction TB
        WtUftEngine["WtUftEngine<br/>UFT引擎<br/>• 策略上下文管理<br/>• 数据订阅管理<br/>• 数据分发<br/>• 时间管理<br/>• 交易日管理"]:::engineClass
        WtUftTicker["WtUftTicker<br/>实时Ticker<br/>• 实时行情处理<br/>• 分钟线闭合判断<br/>• 交易日判断<br/>• 后台定时检查"]:::tickerClass
    end

    %% 策略管理层
    subgraph StrategyMgrs["策略管理层 - 策略生命周期"]
        direction TB
        UftStrategyMgr["UftStrategyMgr<br/>策略管理器<br/>• 加载策略工厂<br/>• 创建策略实例<br/>• 管理策略映射"]:::mgrClass
        UftStraContext["UftStraContext<br/>策略上下文<br/>• 交易接口实现<br/>• 本地持仓管理<br/>• 订单管理<br/>• 数据订阅管理<br/>• 事件转发"]:::contextClass
    end

    %% 数据管理层
    subgraph DataLayer["数据管理层 - 市场数据"]
        direction TB
        WtUftDtMgr["WtUftDtMgr<br/>数据管理器<br/>• 实时Tick缓存<br/>• 历史Tick缓存<br/>• K线缓存<br/>• 数据切片查询"]:::dataClass
    end

    %% 适配器层
    subgraph Adapters["适配器层 - 外部接口适配"]
        direction TB
        TraderAdapter["TraderAdapter<br/>交易适配器<br/>• 交易接口适配<br/>• 订单管理<br/>• 持仓管理<br/>• 动作策略转换<br/>• 风险控制"]:::adapterClass
        ParserAdapter["ParserAdapter<br/>行情解析器适配器<br/>• 行情接口适配<br/>• 数据过滤<br/>• 数据标准化<br/>• 数据转发"]:::adapterClass
    end

    %% 工具支持层
    subgraph Utils["工具支持层 - 辅助功能"]
        direction TB
        EventNotifier["EventNotifier<br/>事件通知器<br/>• 消息队列集成<br/>• 异步事件处理<br/>• JSON格式转换<br/>• 事件广播"]:::utilClass
        ActionPolicyMgr["ActionPolicyMgr<br/>动作策略管理器<br/>• 交易动作规则<br/>• 品种规则映射<br/>• 手数限制管理"]:::utilClass
        ShareManager["ShareManager<br/>共享内存管理器<br/>• 参数共享域管理<br/>• 参数读写<br/>• 参数监控<br/>• 参数同步"]:::utilClass
        WtHelper["WtHelper<br/>辅助工具类<br/>• 路径管理<br/>• 时间管理<br/>• 目录创建"]:::utilClass
    end

    %% 继承关系
    WtUftEngine -.->|"实现"| IParserStub
    UftStraContext -.->|"实现"| IUftStraCtx
    UftStraContext -.->|"实现"| ITrdNotifySink
    WtUftDtMgr -.->|"实现"| IDataManager
    TraderAdapter -.->|"实现"| ITraderSpi
    ParserAdapter -.->|"实现"| IParserSpi

    %% 核心组合关系
    WtUftEngine -->|"包含"| WtUftTicker
    WtUftEngine -->|"管理"| UftStraContext
    WtUftEngine -->|"使用"| WtUftDtMgr
    WtUftEngine -->|"使用"| EventNotifier
    
    UftStrategyMgr -->|"创建"| UftStraContext
    UftStraContext -->|"使用"| TraderAdapter
    UftStraContext -->|"使用"| ShareManager
    UftStraContext -->|"使用数据结构"| UftDataDefs
    
    WtUftTicker -->|"触发事件"| WtUftEngine
    
    %% 数据流关系
    ParserAdapter -->|"推送行情"| WtUftEngine
    WtUftEngine -->|"分发数据"| UftStraContext
    WtUftEngine -->|"更新数据"| WtUftDtMgr
    
    %% 交易流关系
    UftStraContext -->|"下单"| TraderAdapter
    TraderAdapter -->|"交易回报"| UftStraContext
    TraderAdapter -->|"使用"| ActionPolicyMgr
    TraderAdapter -->|"通知事件"| EventNotifier
    
    %% 工具层关系
    WtUftEngine -.->|"使用"| WtHelper
    UftStraContext -.->|"使用"| WtHelper
    WtUftDtMgr -.->|"使用"| WtHelper
    TraderAdapter -.->|"使用"| WtHelper
    ParserAdapter -.->|"使用"| WtHelper
    
    %% 应用样式
    class WtUftEngine engineClass
    class UftStraContext contextClass
    class TraderAdapter,ParserAdapter adapterClass
    class WtUftDtMgr dataClass
    class EventNotifier,ActionPolicyMgr,ShareManager,WtHelper utilClass
    class WtUftTicker tickerClass
    class UftStrategyMgr mgrClass
    class IUftStraCtx,ITrdNotifySink,IParserStub,IDataManager,ITraderSpi,IParserSpi interfaceClass
    class UftDataDefs defClass
```


# 适配器层

## ParserAdapter.h/cpp — 行情解析器适配器

### 框架图
```mermaid
graph LR
    %% 样式定义
    classDef parserInterface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef parserAdapter fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef parserMgr fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef execInterface fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef execUnit fill:#fff9c4,stroke:#f57f17,stroke-width:3px,color:#000;
    classDef execFactory fill:#fce4ec,stroke:#c2185b,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 外部接口层
    %% =======================
    subgraph ExternalLayer["外部接口层"]
        direction TB
        IParserApi["IParserApi<br/>行情解析器API<br/>数据源接口"]
        IParserSpi["IParserSpi<br/>行情解析器回调接口<br/>接收数据回调"]
        WtUftEngine["WtUftEngine<br/>UFT引擎<br/>数据接收者"]:::external
    end

    %% =======================
    %% 行情解析适配器层
    %% =======================
    subgraph ParserLayer["行情解析适配器层 - ParserAdapter"]
        direction TB
        IParserStub["IParserStub<br/>数据推送接口<br/>定义数据接收接口"]:::parserInterface
        ParserAdapter["ParserAdapter<br/>行情解析器适配器<br/>适配不同数据源<br/>数据过滤与转发"]:::parserAdapter
        ParserAdapterMgr["ParserAdapterMgr<br/>适配器管理器<br/>管理多个适配器"]:::parserMgr
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    ParserAdapter -.->|"实现"| IParserSpi
    WtUftEngine -.->|"实现"| IParserStub

    %% =======================
    %% 组合与管理关系（实线）
    %% =======================
    ParserAdapterMgr -->|"管理"| ParserAdapter
    ParserAdapter -->|"使用"| IParserApi
    ParserAdapter -->|"推送数据"| IParserStub
    IParserStub -->|"数据接收者"| WtUftEngine


    %% =======================
    %% 数据流关系
    %% =======================
    IParserApi -->|"数据回调"| ParserAdapter
    ParserAdapter -->|"过滤转发"| WtUftEngine

    %% 应用样式
    class IParserStub parserInterface
    class ParserAdapter parserAdapter
    class ParserAdapterMgr parserMgr
    class ExecuteContext execInterface
    class ExecuteUnit execUnit
    class IExecuterFact execFactory
    class IParserApi,IParserSpi,WtUftEngine external
```

### 解析器存根接口类 IParserStub
```cpp
class IParserStub
```

#### 推送行情数据 handle_push_quote
```cpp
/**
 * @brief 处理Tick数据推送
 * @param curTick 当前Tick数据指针
 * 当解析器接收到Tick数据时调用，推送Tick数据。
 */
virtual void handle_push_quote(WTSTickData* curTick){}
```

#### 推送委托明细数据 handle_push_order_detail
```cpp
/**
 * @brief 处理委托明细数据推送
 * @param curOrdDtl 当前委托明细数据指针
 * 当解析器接收到委托明细数据时调用，推送委托明细数据。
 */
virtual void handle_push_order_detail(WTSOrdDtlData* curOrdDtl){}
```

#### 推送委托队列数据 handle_push_order_queue
```cpp
/**
 * @brief 处理委托队列数据推送
 * @param curOrdQue 当前委托队列数据指针
 * 当解析器接收到委托队列数据时调用，推送委托队列数据。
 */
virtual void handle_push_order_queue(WTSOrdQueData* curOrdQue) {}
```

#### 推送逐笔成交数据 handle_push_transaction
```cpp
/**
 * @brief 处理逐笔成交数据推送
 * @param curTrans 当前逐笔成交数据指针
 * 当解析器接收到逐笔成交数据时调用，推送逐笔成交数据。
 */
virtual void handle_push_transaction(WTSTransData* curTrans) {}
```

### 解析器适配器类 ParserAdapter
```cpp
class ParserAdapter : public IParserSpi,  // 继承解析器SPI接口
	private boost::noncopyable  // 继承boost::noncopyable，禁止拷贝构造和赋值
```

#### 成员
- **核心接口指针**
  - `IParserApi* _parser_api`：行情解析器API指针，用于调用解析器功能（初始化、订阅、连接等）
  - `FuncDeleteParser _remover`：删除解析器函数指针，用于释放动态加载的解析器模块

- **状态标志**
  - `bool _stopped`：是否已停止标志，用于控制数据接收和处理流程

- **数据过滤器**
  - `ExchgFilter _exchg_filter`：交易所过滤器
    - `typedef wt_hashset<std::string> ExchgFilter`：交易所代码集合
    - 仅接收指定交易所的数据
  - `ExchgFilter _code_filter`：合约代码过滤器
    - `typedef wt_hashset<std::string> ExchgFilter`：合约代码或品种代码集合
    - 仅接收指定合约或品种的数据

- **外部依赖指针**
  - `IBaseDataMgr* _bd_mgr`：基础数据管理器指针，用于获取合约信息、查询合约列表等
  - `IParserStub* _stub`：行情数据存根接口指针，用于接收并转发解析后的行情数据（Tick、委托队列、委托明细、逐笔成交）

- **配置与标识**
  - `WTSVariant* _cfg`：配置参数指针，包含解析器模块路径、订阅配置、过滤器设置等
  - `std::string _id`：适配器ID，唯一标识符

#### 初始化与生命周期管理

##### 初始化适配器（从配置文件）init

##### 初始化适配器（外部API）initExt

##### 启动解析器 run
```cpp
/**
 * @brief 启动解析器
 * @return 启动成功返回true，失败返回false
 * 启动行情解析器，开始接收行情数据。
 */
bool ParserAdapter::run()
{
	if (_parser_api == NULL)
		return false;
	_parser_api->connect();
	return true;
}
```

#### IParserSpi接口回调 - 行情数据回调

##### 处理合约列表 handleSymbolList
```cpp
/**
 * @brief 处理合约列表（IParserSpi接口）
 * @param aySymbols 合约列表数组
 * 当解析器返回合约列表时调用。默认实现为空。
 */
virtual void handleSymbolList(const WTSArray* aySymbols) override {}
```

##### 处理实时行情（Tick数据）handleQuote
当解析器接收到Tick行情数据时调用：
- 验证数据有效性，获取合约信息，标准化合约代码，转发给存根接口 `_stub`
```cpp
/**
 * @brief 处理实时行情（IParserSpi接口）
 * @param quote 实时行情数据指针
 * @param procFlag 处理标志，是否需要切片

 */
void ParserAdapter::handleQuote(WTSTickData *quote, uint32_t procFlag)
{
	if (quote == NULL || _stopped || quote->actiondate() == 0)
		return;

	WTSContractInfo* cInfo = quote->getContractInfo();
	if (cInfo == NULL) cInfo = _bd_mgr->getContract(quote->code(), quote->exchg());
	if (cInfo == NULL)
		return;

	quote->setCode(cInfo->getFullCode()); // 设置标准化合约代码
	_stub->handle_push_quote(quote); // 转发Tick数据给存根接口
}
```

##### 处理委托队列数据（股票level2）handleOrderQueue
当解析器接收到委托队列数据时调用，用于股票level2行情：
- 验证数据有效性，检查交易所过滤器，获取合约信息，标准化合约代码，转发给存根接口 `_stub`
```cpp
/**
 * @brief 处理委托队列数据（IParserSpi接口，股票level2）
 * @param ordQueData 委托队列数据指针
 */
void ParserAdapter::handleOrderQueue(WTSOrdQueData* ordQueData)
{
	if (_stopped)
		return;
	if (!_exchg_filter.empty() && (_exchg_filter.find(ordQueData->exchg()) == _exchg_filter.end()))
		return;
	if (ordQueData->actiondate() == 0 || ordQueData->tradingdate() == 0)
		return;
	WTSContractInfo* cInfo = _bd_mgr->getContract(ordQueData->code(), ordQueData->exchg());
	if (cInfo == NULL)
		return;

	ordQueData->setCode(cInfo->getFullCode());
	if (_stub)
		_stub->handle_push_order_queue(ordQueData);
}
```

##### 处理逐笔委托数据（股票level2）handleOrderDetail
当解析器接收到逐笔委托数据时调用，用于股票level2行情:
- 验证数据有效性，检查交易所过滤器，获取合约信息，标准化合约代码，转发给存根接口 `_stub`
```cpp
/**
 * @brief 处理逐笔委托数据（IParserSpi接口，股票level2）
 * @param ordDtlData 逐笔委托数据指针
 */
void ParserAdapter::handleOrderDetail(WTSOrdDtlData* ordDtlData)
{
	if (_stopped)
		return;
	if (!_exchg_filter.empty() && (_exchg_filter.find(ordDtlData->exchg()) == _exchg_filter.end()))
		return;
	if (ordDtlData->actiondate() == 0 || ordDtlData->tradingdate() == 0)
		return;
	WTSContractInfo* cInfo = _bd_mgr->getContract(ordDtlData->code(), ordDtlData->exchg());
	if (cInfo == NULL)
		return;

	ordDtlData->setCode(cInfo->getFullCode());
	if (_stub)
		_stub->handle_push_order_detail(ordDtlData);
}
```

##### 处理逐笔成交数据 handleTransaction
当解析器接收到逐笔成交数据时调用。
- 验证数据有效性，检查交易所过滤器，获取合约信息，标准化合约代码，转发给存根接口 `_stub`
```cpp
/**
 * @brief 处理逐笔成交数据（IParserSpi接口）
 * @param transData 逐笔成交数据指针
 */
void ParserAdapter::handleTransaction(WTSTransData* transData)
{
	if (_stopped)
		return;
	if (!_exchg_filter.empty() && (_exchg_filter.find(transData->exchg()) == _exchg_filter.end()))
		return;
	if (transData->actiondate() == 0 || transData->tradingdate() == 0)
		return;
	WTSContractInfo* cInfo = _bd_mgr->getContract(transData->code(), transData->exchg());
	if (cInfo == NULL)
		return;

	transData->setCode(cInfo->getFullCode());
	if (_stub)  // 如果存根接口存在
		_stub->handle_push_transaction(transData);
}
```

#### 辅助方法

##### 处理解析器日志 handleParserLog
```cpp
/**
 * @brief 处理解析器日志（IParserSpi接口）
 * @param ll 日志级别
 * @param message 日志消息
 * 当解析器输出日志时调用，转发日志到日志系统。
 */
void ParserAdapter::handleParserLog(WTSLogLevel ll, const char* message)
{
	if (_stopped)
		return;
	WTSLogger::log_dyn_raw("parser", _id.c_str(), ll, message);
}
```

##### 获取基础数据管理器 getBaseDataMgr
```cpp
/**
 * @brief 获取基础数据管理器（IParserSpi接口）
 * @return 基础数据管理器指针
 * 返回基础数据管理器，供解析器查询合约信息等。
 */
virtual IBaseDataMgr* getBaseDataMgr() override { return _bd_mgr; }
```

### 解析器适配器管理器类 ParserAdapterMgr
```cpp
class ParserAdapterMgr : private boost::noncopyable  // 继承boost::noncopyable，禁止拷贝构造和赋值
```

#### 成员
`ParserAdapterMap _adapters`：解析器适配器映射表，键为解析器ID，值为适配器指针
- typedef wt_hashmap\<std::string, `ParserAdapterPtr`\> ParserAdapterMap
- typedef std::shared_ptr\<`ParserAdapter`\> ParserAdapterPtr

#### 方法

##### 添加适配器 addAdapter
```cpp
/**
 * @brief 添加适配器
 * @param id 适配器ID
 * @param adapter 适配器智能指针
 * @return 添加成功返回true，失败返回false
 * 将适配器添加到管理器中。如果ID已存在，则添加失败。
 */
bool ParserAdapterMgr::addAdapter(const char* id, ParserAdapterPtr& adapter)
{
	if (adapter == NULL || strlen(id) == 0)
		return false;
	auto it = _adapters.find(id);
	if (it != _adapters.end())
	{
		WTSLogger::error(" Same name of parsers: {}", id);
		return false;
	}

	_adapters[id] = adapter;
	return true;
}
```

##### 获取适配器 getAdapter
```cpp
/**
 * @brief 获取适配器
 * @param id 适配器ID
 * @return 适配器智能指针，如果不存在返回空指针
 * 根据适配器ID查找对应的适配器。
 */
ParserAdapterPtr ParserAdapterMgr::getAdapter(const char* id)
{
	auto it = _adapters.find(id);
	if (it != _adapters.end())
	{
		return it->second;
	}
	return ParserAdapterPtr();
}
```

##### 启动所有适配器 run
```cpp
/**
 * @brief 启动所有适配器
 * 启动所有管理的适配器，开始接收行情数据。
 */
void ParserAdapterMgr::run()
{
	for (auto it = _adapters.begin(); it != _adapters.end(); it++)
	{
		it->second->run();
	}
	WTSLogger::info("{} parsers started", _adapters.size());
}
```

## TraderAdapter.h/cpp — 交易适配器

### 框架图
```mermaid
graph LR
    %% 样式定义
    classDef traderInterface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef traderAdapter fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef traderMgr fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef notifyInterface fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef policyMgr fill:#fce4ec,stroke:#c2185b,stroke-width:2px,color:#000;
    classDef context fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 外部接口层
    %% =======================
    subgraph ExternalLayer["外部接口层"]
        direction TB
        ITraderApi["ITraderApi<br/>交易接口API<br/>交易通道接口"]
        ITraderSpi["ITraderSpi<br/>交易接口回调接口<br/>接收交易回调"]
        UftStraContext["UftStraContext<br/>策略上下文<br/>交易请求者"]:::context
    end

    %% =======================
    %% 交易适配器层
    %% =======================
    subgraph TraderLayer["交易适配器层 - TraderAdapter"]
        direction TB
        ITrdNotifySink["ITrdNotifySink<br/>交易通知接口<br/>定义交易事件接收接口"]:::notifyInterface
        TraderAdapter["TraderAdapter<br/>交易适配器<br/>适配不同交易接口<br/>订单与持仓管理<br/>风险控制"]:::traderAdapter
        TraderAdapterMgr["TraderAdapterMgr<br/>适配器管理器<br/>管理多个适配器"]:::traderMgr
    end

    %% =======================
    %% 工具支持层
    %% =======================
    subgraph UtilsLayer["工具支持层"]
        direction TB
        ActionPolicyMgr["ActionPolicyMgr<br/>动作策略管理器<br/>交易动作规则管理"]:::policyMgr
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    TraderAdapter -.->|"实现"| ITraderSpi
    UftStraContext -.->|"实现"| ITrdNotifySink

    %% =======================
    %% 组合与管理关系（实线）
    %% =======================
    TraderAdapterMgr -->|"管理"| TraderAdapter
    TraderAdapter -->|"使用"| ITraderApi
    TraderAdapter -->|"使用"| ActionPolicyMgr
    TraderAdapter -->|"通知"| ITrdNotifySink
    ITrdNotifySink -->|"数据接收者"| UftStraContext

    %% =======================
    %% 交易流关系
    %% =======================
    UftStraContext -->|"下单请求"| TraderAdapter
    ITraderApi -->|"交易回调"| TraderAdapter
    TraderAdapter -->|"交易回报"| UftStraContext

    %% 应用样式
    class ITrdNotifySink notifyInterface
    class TraderAdapter traderAdapter
    class TraderAdapterMgr traderMgr
    class ActionPolicyMgr policyMgr
    class UftStraContext context
    class ITraderApi,ITraderSpi external
```

### 交易适配器类 TraderAdapter
```cpp
class TraderAdapter : public ITraderSpi
```

#### 成员
- **配置与标识**
  - `WTSVariant* _cfg`：配置参数
  - `std::string _id`：适配器ID
  - `std::string _order_pattern`：订单用户标签模式
  - `uint32_t _trading_day`：交易日
- **核心接口指针**
  - `ITraderApi* _trader_api`：交易接口指针
  - `FuncDeleteTrader _remover`：删除交易接口的函数指针
- **状态管理**
  - `AdapterState _state`：适配器状态（枚举类型）
    ```cpp
    /* 定义交易通道从连接、登录到就绪的各个状态*/
    typedef enum tagAdapterState
    {
      AS_NOTLOGIN, // 未登录状态：初始状态，尚未开始登录流程
      AS_LOGINING, // 正在登录：已发起登录请求，等待登录结果
      AS_LOGINED, // 已登录：登录成功，但尚未完成数据查询
      AS_LOGINFAILED, // 登录失败：登录请求被拒绝或失败
      AS_POSITION_QRYED, // 仓位已查：持仓查询完成
      AS_ORDERS_QRYED, // 订单已查：订单查询完成
      AS_TRADES_QRYED, // 成交已查：成交查询完成
      AS_ALLREADY // 全部就绪：所有查询完成，交易通道可以使用
    }
    ```
- **外部依赖指针**
  - `wt_hashset<ITrdNotifySink*> _sinks`：通知接收器集合
  - `IBaseDataMgr* _bd_mgr`：基础数据管理器指针
  - `ActionPolicyMgr* _policy_mgr`：动作策略管理器指针
- **持仓管理**
  - `wt_hashmap<std::string, PosItem> _positions`：持仓映射表，键为合约代码
    ```cpp
    /* @brief 持仓项结构体
      * 用于存储单个合约的持仓信息，包括多空两个方向的今昨持仓数据*/
    typedef struct _PosItem
    {
      // 多仓数据（做多方向持仓）
      double l_newvol; // 多头今仓数量：今日开仓的多头持仓数量
      double l_newavail; // 多头今仓可用：今日开仓的多头持仓中可用于平仓的数量
      double l_prevol; // 多头昨仓数量：昨日及之前开仓的多头持仓数量
      double l_preavail; // 多头昨仓可用：昨日及之前开仓的多头持仓中可用于平仓的数量
      // 空仓数据（做空方向持仓）
      double s_newvol; // 空头今仓数量：今日开仓的空头持仓数量
      double s_newavail; // 空头今仓可用：今日开仓的空头持仓中可用于平仓的数量
      double s_prevol; // 空头昨仓数量：昨日及之前开仓的空头持仓数量
      double s_preavail; // 空头昨仓可用：昨日及之前开仓的空头持仓中可用于平仓的数量
    } PosItem;
    ```
- **订单管理**
  - `SpinMutex _mtx_orders`：订单列表互斥锁
  - `OrderMap* _orders`：订单映射表
    - `typedef WTSMap<uint32_t> OrderMap`：订单映射表类型
  - `wt_hashset<std::string> _orderids`：订单号集合，主要用于标记是否处理过该订单
  - `wt_hashmap<std::string, double> _undone_qty`：未完成数量映射表，键为合约代码
- **交易统计**
  - `TradeStatMap* _stat_map`：交易统计映射表，键为合约代码
    - `typedef WTSHashMap<std::string> TradeStatMap`：交易统计映射表类型
- **风险控制时间缓存**
  - `CodeTimeCacheMap _order_time_cache`：下单时间缓存，键为合约代码，值为时间戳列表
  - `CodeTimeCacheMap _cancel_time_cache`：撤单时间缓存，键为合约代码，值为时间戳列表
    - `typedef std::vector<uint64_t> TimeCacheList`：时间缓存列表类型
    - `typedef wt_hashmap<std::string, TimeCacheList> CodeTimeCacheMap`：代码时间缓存映射表类型
- **风险控制**
  - `wt_hashset<std::string> _exclude_codes`：被风控排除的合约代码集合
  - `RiskParamsMap _risk_params_map`：风险参数映射表，键为品种代码
    - `typedef wt_hashmap<std::string, RiskParams> RiskParamsMap`：风险参数映射表类型
      ```cpp
      /* @brief 风控参数结构体
      * 定义交易风控策略的参数，包括下单和撤单的频率限制 */
      typedef struct _RiskParams
      {
        uint32_t _order_times_boundary; // 下单频率边界：在统计时间窗口内允许的最大下单次数
        uint32_t _order_stat_timespan; // 下单统计时间窗口：统计下单频率的时间跨度（秒）
        uint32_t _order_total_limits; // 下单总限额：当日允许的最大下单总次数

        uint32_t _cancel_times_boundary; // 撤单频率边界：在统计时间窗口内允许的最大撤单次数
        uint32_t _cancel_stat_timespan; // 撤单统计时间窗口：统计撤单频率的时间跨度（秒）
        uint32_t _cancel_total_limits; // 撤单总限额：当日允许的最大撤单总次数
      } RiskParams;
      ```
  - `bool _risk_mon_enabled`：是否启用风险监控

#### 初始化与生命周期管理

##### 初始化交易适配器（从配置文件）init

##### 初始化交易适配器（外部API）initExt

##### 启动交易适配器 run
```cpp
/**
 * @brief 启动交易适配器
 * @return 启动成功返回true，失败返回false
 * 注册回调接口，连接交易服务器，开始登录流程。
 */
bool TraderAdapter::run()
{
	if (_trader_api == NULL)
		return false;
	if (_stat_map == NULL)
		_stat_map = TradeStatMap::create();

	_trader_api->registerSpi(this); // 注册回调接口
	_trader_api->connect(); // 连接交易服务器
	_state = AS_LOGINING; // 设置状态为正在登录
	return true;
}
```

#### 持仓和订单管理

##### 获取持仓数量 getPosition
获取指定合约的持仓数量：
* **查找持仓记录**
  * 在内部持仓映射表 `_positions` 中查找指定的 `stdCode`。如果未找到，直接返回 0。
* **计算多头持仓**
  * 检查 `flag` 是否包含多头标志（`flag & 1`）。
  * 如果包含，根据 `bValidOnly` 判断：
    * **可用持仓**：累加今仓可用 (`l_newavail`) 和昨仓可用 (`l_preavail`)。
    * **全部持仓**：累加今仓总量 (`l_newvol`) 和昨仓总量 (`l_prevol`)。
* **计算空头持仓**
  * 检查 `flag` 是否包含空头标志（`flag & 2`）。
  * 如果包含，根据 `bValidOnly` 判断，并**做减法**（空头记为负数）：
    * **可用持仓**：减去今仓可用 (`s_newavail`) 和昨仓可用 (`s_preavail`)。
    * **全部持仓**：减去今仓总量 (`s_newvol`) 和昨仓总量 (`s_prevol`)。
* **返回结果**
  * 返回计算后的净持仓数量（多头为正，空头为负）。
```cpp
/**
 * @brief 获取持仓数量
 * @param stdCode 标准合约代码
 * @param bValidOnly 是否只返回可用持仓，true表示只返回可用持仓，false表示返回全部持仓
 * @param flag 持仓标志：1-多仓，2-空仓，3-全部（默认）
 * @return 持仓数量，多仓为正，空仓为负
 */
double TraderAdapter::getPosition(const char* stdCode, bool bValidOnly, int32_t flag /* = 3 */)
```

##### 枚举持仓并通知接收器 enumPosition
遍历持仓并通知监听器 `_sinks`：
* **初始化与筛选**
  * 判断是否枚举所有合约：如果 `stdCode` 为空字符串，则标记 `bAll` 为 true。
* **遍历持仓映射表**
  * 遍历内部的 `_positions` 容器。
  * **过滤逻辑**：如果不是枚举所有合约且当前合约代码与 `stdCode` 不匹配，则跳过。
* **推送通知**
  * 获取当前合约的持仓项 `pItem`。
  * 遍历所有注册的通知接收器 `_sinks`：
    * **通知多头**：调用 `sink->on_position`，传入多头标志 true，以及多头的昨仓/今仓总量和可用量。
    * **通知空头**：调用 `sink->on_position`，传入多头标志 false，以及空头的昨仓/今仓总量和可用量。
* **统计总持仓**
  * 累加当前合约的多头总持仓和空头总持仓到返回值 `ret` 中，最终返回总持仓数。
```cpp
/**
 * @brief 枚举持仓并通知接收器
 * @param stdCode 标准合约代码，空字符串表示枚举所有合约
 * @return 总持仓数量
 * * 遍历持仓，通过回调函数通知所有接收器。
 * 使用回调方式，避免接口设计过于复杂。
 */
double TraderAdapter::enumPosition(const char* stdCode /* = "" */)
```

##### 获取订单列表 getOrders
获取订单列表的快照：
* **前置检查**
  * 检查内部订单容器 `_orders` 是否为空，如果为空直接返回 NULL。
* **线程安全与创建**
  * 使用自旋锁 `SpinLock lock(_mtx_orders)` 锁定订单容器，防止在读取时发生数据竞争。
  * 创建一个新的订单映射表 `OrderMap` 对象 `ret` 用于存储结果。
* **遍历与过滤**
  * 遍历内部订单容器 `_orders`。
  * **过滤逻辑**：如果请求所有合约 (`stdCode` 为空) 或者当前订单的代码与 `stdCode` 匹配：
    * 将该订单的本地 ID (`localid`) 和订单信息 (`ordInfo`) 添加到结果集 `ret` 中。
* **返回结果**
  * 返回新创建的 `OrderMap` 指针（调用者后续需要负责释放该对象的内存）。
```cpp
/**
 * @brief 获取订单列表
 * @param stdCode 标准合约代码，空字符串表示获取所有订单
 * @return 订单映射表指针，调用者需要释放
 */
OrderMap* TraderAdapter::getOrders(const char* stdCode)
```

##### 获取未完成数量 getUndoneQty
```cpp
/**
 * @brief 获取未完成数量
 * @param stdCode 标准合约代码
 * @return 未完成数量
 * 获取指定合约的未完成订单数量。
 */
inline double getUndoneQty(const char* stdCode)
{
    auto it = _undone_qty.find(stdCode);
    if (it != _undone_qty.end())
        return it->second;
    return 0;
}
```

##### 获取交易统计信息数量 getInfos
```cpp
/**
 * @brief 获取交易统计信息数量
 * @param stdCode 标准合约代码
 * @return 统计信息数量
 */
uint32_t TraderAdapter::getInfos(const char* stdCode)
{
	WTSTradeStateInfo* statInfo = (WTSTradeStateInfo*)_stat_map->get(stdCode);
	if (statInfo == NULL)
		return 0;
	return statInfo->infos();
}
```

#### 交易操作

##### 买入操作 buy
该函数用于执行**买入操作**：根据预设的策略规则（`ActionPolicyMgr`），将一个高层的 *买入* 信号转换为实际的交易指令。这些指令可能是**开多仓**，也可能是**平空仓（含平昨/平今）**，具体取决于当前的持仓状态和配置的策略规则。
* **初始化与准备**
  * **参数检查**：检查请求数量 `qty` 是否为 0，若为 0 直接返回空列表
  * **获取信息**：获取合约信息（`cInfo`）、品种信息（`commInfo`）、当前持仓信息（`pItem`）以及交易统计信息（`statItem`）
  * **获取策略规则**：根据品种 ID 从策略管理器 `_policy_mgr` 中获取对应的动作规则组 `ActionRuleGroup ruleGP`
  * **拆单阈值**：获取单笔最大委托数量 (`unitQty`)，用于后续大单拆分

* **遍历策略规则执行交易**
  * 函数遍历规则组 `ruleGP` 中的每一条规则，直到目标数量 `left` 归零。根据规则类型 (`_atype`) 不同，执行不同逻辑：

  * **A. 开仓规则 (`AT_Open`)**
    * 如果规则是开仓且非强制平仓（**开多**）：
      1. **限额检查**：检查该合约今日多头开仓量或总开仓量是否已达上限 (`_limit_l` 或 `_limit`)。如果超限，跳过该规则。
      2. **计算数量**：取下面的最小值作为本次下单量
         1. 剩余目标数量 `left`
         2. 如果多仓限额 `_limit_l` 非 0，`_limit_l - l_openvol`
         3. 如果总限额 `_limit` 非 0，`_limit - l_openvol - s_openvol`
      3. **拆单与执行**：将下单量按 `unitQty` 拆分，循环调用 `openLong` 发出开多指令，并将生成的订单 ID 加入返回列表。
  * **B. 平今规则 (`AT_CloseToday`) -> 平今空**
    1. **确定可平量**：
       * 若品种区分平昨平今 (`CM_CoverToday`)，取今仓可用量 (`s_newavail`)。
       * 若不区分，取所有空头可用量。
    2. **净仓检查**：如果规则要求净仓 (`_pure`) 且昨仓不为 0，跳过此规则（避免锁仓）。
       *  **避免 *留老平新*：只要还有老单子，就禁止平新单子**
    3. **执行**：调用 `closeShort`，若支持平今则标记 `isToday=true`。

  * **C. 平昨规则 (`AT_CloseYestoday`) -> 平昨空**
    1. **确定可平量**：取昨仓可用量 (`s_preavail`)。
    2. **净仓检查**：如果规则要求净仓且今仓不为 0，跳过此规则。
       *  **避免 *留新平老*：只要还有新单子，就禁止平老单子**
    3. **执行**：调用 `closeShort`，标记 `isToday=false`。
  * **D. 平仓规则 (`AT_Close`) -> 智能平空**
    * 根据品种是否区分平昨平今采取不同策略：
    * **不区分平昨平今**：直接获取所有空头可用量，调用 `closeShort`。
    * **区分平昨平今**：
      1. **优先平昨**：先检查昨仓可用量 (`s_preavail`)，调用 `closeShort`（`isToday=false`）。
      2. **由昨转今**：如果还有剩余目标数量 (`left > 0`)，再检查今仓可用量 (`s_newavail`)，调用 `closeShort`（`isToday=true`）。
* **循环结束与收尾**
  * **退出条件**：在遍历过程中，一旦剩余目标数量 `left` 变为 0，立即退出循环。
  * **异常记录**：如果所有规则遍历完后仍有 `left > 0`，记录错误日志，提示剩余数量未被触发。
  * **返回结果**：返回所有发出的本地订单 ID 列表 (`OrderIDs`)。

```cpp
/**
 * @brief 买入操作（根据策略规则转换为实际订单）
 * @param stdCode 标准合约代码
 * @param price 价格，0表示市价单
 * @param qty 数量
 * @param flag 下单标志：0-normal，1-fak，2-fok
 * @param bForceClose 是否强制平仓
 * @param cInfo 合约信息指针，可为NULL
 * @return 订单ID列表
 */
OrderIDs TraderAdapter::buy(const char* stdCode, double price, double qty, int flag, bool bForceClose, WTSContractInfo* cInfo /* = NULL */)
```

##### 卖出操作 sell
该函数用于执行**卖出操作**：根据预设的策略规则（`ActionPolicyMgr`），将一个高层的 *卖出* 信号转换为实际的交易指令。这些指令可能是**开空仓**，也可能是**平多仓（含平昨/平今）**，具体取决于当前的持仓状态和配置的策略规则。
* **初始化与准备**
  * **参数检查**：检查请求数量 `qty` 是否为 0，若为 0 直接返回空列表
  * **获取信息**：获取合约信息（`cInfo`）、品种信息（`commInfo`）、当前持仓信息（`pItem`）以及交易统计信息（`statItem`）
  * **获取策略规则**：根据品种 ID 从策略管理器 `_policy_mgr` 中获取对应的动作规则组 `ActionRuleGroup ruleGP`
  * **拆单阈值**：获取单笔最大委托数量 (`unitQty`)，用于后续大单拆分
* **遍历策略规则执行交易**
  * 函数遍历规则组 `ruleGP` 中的每一条规则，直到目标数量 `left` 归零。根据规则类型 (`_atype`) 不同，执行不同逻辑：
  * **A. 开仓规则 (`AT_Open`) -> 开空**
    * 如果规则是开仓且非强制平仓（**开空**）：
      1. **限额检查**：检查该合约今日空头开仓量或总开仓量是否已达上限 (`_limit_s` 或 `_limit`)。如果超限，跳过该规则。
      2. **计算数量**：取下面的最小值作为本次下单量
         1. 剩余目标数量 `left`
         2. 如果空仓限额 `_limit_s` 非 0，`_limit_s - s_openvol`
         3. 如果总限额 `_limit` 非 0，`_limit - l_openvol - s_openvol`
      3. **拆单与执行**：将下单量按 `unitQty` 拆分，循环调用 `openShort` 发出开空指令，并将生成的订单 ID 加入返回列表。
  * **B. 平今规则 (`AT_CloseToday`) -> 平今多**
    1. **确定可平量**：
       * 若品种区分平昨平今 (`CM_CoverToday`)，取多头今仓可用量 (`l_newavail`)。
       * 若不区分，取所有多头可用量。
    2. **净仓检查**：如果规则要求净仓 (`_pure`) 且昨仓不为 0，跳过此规则（避免锁仓）。
       * **避免 *留老平新*：只要还有老单子，就禁止平新单子**
    3. **执行**：调用 `closeLong`，若支持平今则标记 `isToday=true`。
  * **C. 平昨规则 (`AT_CloseYestoday`) -> 平昨多**
    1. **确定可平量**：取多头昨仓可用量 (`l_preavail`)。
    2. **净仓检查**：如果规则要求净仓且今仓不为 0，跳过此规则。
       * **避免 *留新平老*：只要还有新单子，就禁止平老单子**
    3. **执行**：调用 `closeLong`，标记 `isToday=false`。
  * **D. 平仓规则 (`AT_Close`) -> 智能平多**
    * 根据品种是否区分平昨平今采取不同策略：
    * **不区分平昨平今**：直接获取所有多头可用量，调用 `closeLong`。
    * **区分平昨平今**：
      1. **优先平昨**：先检查多头昨仓可用量 (`l_preavail`)，调用 `closeLong`（`isToday=false`）。
      2. **由昨转今**：如果还有剩余目标数量 (`left > 0`)，再检查多头今仓可用量 (`l_newavail`)，调用 `closeLong`（`isToday=true`）。
* **循环结束与收尾**
  * **退出条件**：在遍历过程中，一旦剩余目标数量 `left` 变为 0，立即退出循环。
  * **异常记录**：如果所有规则遍历完后仍有 `left > 0`，记录错误日志，提示剩余数量未被触发。
  * **返回结果**：返回所有发出的本地订单 ID 列表 (`OrderIDs`)。
```cpp
/**
 * @brief 卖出操作（根据策略规则转换为实际订单）
 * @param stdCode 标准合约代码
 * @param price 价格，0表示市价单
 * @param qty 数量
 * @param flag 下单标志：0-normal，1-fak，2-fok
 * @param bForceClose 是否强制平仓
 * @param cInfo 合约信息指针，可为NULL
 * @return 订单ID列表
 * * 根据动作策略规则，将卖出信号转换为实际的开空或平多订单。
 */
OrderIDs TraderAdapter::sell(const char* stdCode, double price, double qty, int flag, bool bForceClose, WTSContractInfo* cInfo /* = NULL */)
```

##### 开多单 openLong
该函数用于**买入开仓（做多）**。它构建底层的委托单对象，设置特定的开仓标志和方向，并发送给交易接口：
* **委托构建**
  * 创建 `WTSEntrust` 对象，设置合约代码 `stdCode`、数量 `qty` 和价格 `price`。
  * **价格类型处理**：
    * 如果 `price` 为 0.0，设置价格类型为**市价单** (`WPT_ANYPRICE`)。
    * 否则，设置价格类型为**限价单** (`WPT_LIMITPRICE`)。
  * **订单标志**：根据 `flag` 设置订单标志（如 FAK/FOK），默认为普通单 (`WOF_NOR`)。
* **方向与开平设置**
  * 设置交易方向为**多头** (`WDT_LONG`)。
  * 设置开平类型为**开仓** (`WOT_OPEN`)。
* **状态更新与执行**
  * 调用 `updateUndone` 增加该合约的未完成单数量（`_undone_qty`）。
  * 调用 `doEntrust` 执行委托，生成并返回本地订单 ID (`localid`)。
```cpp
/**
 * @brief 开多单
 * @param stdCode 标准合约代码
 * @param price 价格，0表示市价单
 * @param qty 数量
 * @param flag 下单标志：0-normal，1-fak，2-fok，默认0
 * @return 本地订单ID，失败返回0
 */
uint32_t TraderAdapter::openLong(const char* stdCode, double price, double qty, int flag /* = 0 */)
```

##### 开空单 openShort
该函数用于**卖出开仓（做空）**。逻辑与开多单类似，核心区别在于方向的设置：
* **委托构建**
  * 创建 `WTSEntrust` 对象，设置代码、数量。
  * 根据价格是否为 0.0 选择**市价**或**限价**模式。
  * 设置订单标志（`flag`）。
* **方向与开平设置**
  * 设置交易方向为**空头** (`WDT_SHORT`)。
  * 设置开平类型为**开仓** (`WOT_OPEN`)。
* **状态更新与执行**
  * 调用 `updateUndone` 增加未完成数量。
  * 调用 `doEntrust` 执行委托并返回本地 ID。
```cpp
/**
 * @brief 开空单
 * @param stdCode 标准合约代码
 * @param price 价格，0表示市价单
 * @param qty 数量
 * @param flag 下单标志：0-normal，1-fak，2-fok，默认0
 * @return 本地订单ID，失败返回0
 */
uint32_t TraderAdapter::openShort(const char* stdCode, double price, double qty, int flag/* = 0*/)
```

##### 平多单 closeLong
该函数用于**卖出平仓（平多）**。除了基本的委托构建外，它支持区分**平今**和**平昨**（普通平仓）：
* **委托构建**
  * 创建 `WTSEntrust` 对象。
  * 根据价格参数判断是**市价**还是**限价**。
* **方向与开平设置**
  * 设置交易方向为**多头** (`WDT_LONG`)（注：此处方向通常指操作的持仓方向，配合 Close 使用）。
  * **平今判断**：根据参数 `isToday`：
    * 若为 `true`，设置开平类型为**平今** (`WOT_CLOSETODAY`)。
    * 若为 `false`，设置开平类型为**平仓** (`WOT_CLOSE`)。
* **状态更新与执行**
  * 调用 `updateUndone` 增加未完成数量。
  * 调用 `doEntrust` 发送平仓指令。
```cpp
/**
 * @brief 平多单
 * @param stdCode 标准合约代码
 * @param price 价格，0表示市价单
 * @param qty 数量
 * @param isToday 是否平今仓，默认false
 * @param flag 下单标志：0-normal，1-fak，2-fok，默认0
 * @return 本地订单ID，失败返回0
 */
uint32_t TraderAdapter::closeLong(const char* stdCode, double price, double qty, bool isToday /* = false */, int flag/* = 0*/)
```

##### 平空单 closeShort
该函数用于**买入平仓（平空）**。逻辑与平多单类似，主要针对空头持仓进行操作：
* **委托构建**
  * 创建 `WTSEntrust` 对象，设置价格类型（市价/限价）和订单标志。
* **方向与开平设置**
  * 设置交易方向为**空头** (`WDT_SHORT`)。
  * **平今判断**：根据参数 `isToday`：
    * `true` 对应 `WOT_CLOSETODAY`（平今）。
    * `false` 对应 `WOT_CLOSE`（平仓）。
* **状态更新与执行**
  * 调用 `updateUndone` 增加未完成数量。
  * 调用 `doEntrust` 发送平仓指令。
```cpp
/**
 * @brief 平空单
 * @param stdCode 标准合约代码
 * @param price 价格，0表示市价单
 * @param qty 数量
 * @param isToday 是否平今仓，默认false
 * @param flag 下单标志：0-normal，1-fak，2-fok，默认0
 * @return 本地订单ID，失败返回0
 */
uint32_t TraderAdapter::closeShort(const char* stdCode, double price, double qty, bool isToday /* = false */, int flag/* = 0*/)
```

##### 撤单 cancel
用于**撤销指定订单**。它根据本地订单 ID 查找订单信息并发送撤单请求：
* **查找订单**
  * 检查订单列表是否为空。
  * 加锁并从 `_orders` 映射表中根据 `localid` 查找 `WTSOrderInfo` 对象。
  * 如果订单不存在，返回 `false`。
* **执行撤单**
  * 调用内部函数 `doCancel`：
    * 检查订单是否处于活跃状态 (`isAlive`)。
    * 获取合约信息。
    * 构建撤单动作 `WTSEntrustAction`，设置 `EntrustID` 和 `OrderID`。
    * 调用底层接口 `_trader_api->orderAction` 发送请求。
* **资源释放**
  * 释放获取到的订单信息对象引用，并返回操作结果。
```cpp
/**
 * @brief 撤单
 * @param localid 本地订单ID
 * @return 撤单成功返回true，失败返回false
 */
bool TraderAdapter::cancel(uint32_t localid)
```

##### 全部撤单 cancelAll
用于**批量撤单**。它可以撤销所有订单，或撤销指定合约代码的所有订单：
* **初始化**
  * 判断是否指定了合约代码（`stdCode` 为空字符串则表示全部撤销）。
* **遍历订单**
  * 检查订单列表是否为空。
  * 遍历 `_orders` 容器中的所有订单。
* **筛选与撤单**
  * **活跃检查**：跳过已经结束 (`!isAlive`) 的订单。
  * **代码匹配**：如果指定了 `stdCode`，仅撤销合约代码匹配的订单。
  * **执行撤单**：对符合条件的订单调用 `doCancel`。
  * **记录结果**：如果撤单请求发送成功，将该订单的 `localid` 加入返回列表。
* **返回**
  * 返回成功发送撤单请求的订单 ID 列表 (`OrderIDs`)。
```cpp
/**
 * @brief 全部撤单
 * @param stdCode 标准合约代码，空字符串表示撤所有订单
 * @return 订单ID列表
 */
OrderIDs TraderAdapter::cancelAll(const char* stdCode)
```

#### 风险控制

##### 检查合约是否允许交易 isTradeEnabled
用于**检查合约是否允许交易**：
* **风控开关检查**
  * 首先检查全局风控开关 `_risk_mon_enabled`。
  * 如果未开启风控（`false`），则直接“放行”，返回 `true`（允许交易）。
* **黑名单检查**
  * 如果风控已开启，检查该合约代码 `stdCode` 是否存在于 `_exclude_codes`（禁止交易名单/黑名单）中。
  * **如果在名单中**：返回 `false`（禁止交易）。
  * **如果不在名单中**：返回 `true`（允许交易）。
```cpp
/**
 * @brief 检查合约是否允许交易
 * @param stdCode 标准合约代码
 * @return 允许交易返回true，禁止交易返回false
 */
bool TraderAdapter::isTradeEnabled(const char* stdCode) const
```

##### 检查撤单限制 checkCancelLimits
用于**检查撤单限制**。它在每次撤单前被调用，用于监测该合约的撤单行为是否触发了预设的风控阈值（总次数限制或高频撤单限制）。如果触发，会将该合约 *拉黑*。

* **黑名单预检**
  * 如果合约已经在 `_exclude_codes` 中，直接返回 `false`。
* **获取风控参数**
  * 调用 `getRiskParams` 获取该合约对应的风控配置（如最大撤单次数、统计时间窗口等）。如果没有配置，则默认允许，返回 `true`。
* **总撤单次数检查**
  * 获取该合约的交易统计信息 `statInfo`。
  * 如果配置了总限额 (`_cancel_total_limits != 0`) 且 **当前累计撤单次数 >= 总限额**：
    * 记录错误日志。
    * **拉黑**：将该合约加入 `_exclude_codes`。
    * 返回 `false`。
* **撤单频率检查（流量控制）**
  * 获取该合约的撤单时间戳缓存 `_cancel_time_cache`。
  * 如果缓存中的记录数量达到了边界值 (`_cancel_times_boundary`)：
    * **计算时间窗口**：取当前（最新）记录的时间戳，向前推 `_cancel_stat_timespan` 秒，得到起始时间 `sTime`。
    * **统计区间次数**：使用 `std::lower_bound` 查找起始时间在缓存中的位置，计算该时间窗口内的实际撤单次数。
    * **判定**：如果 **区间内撤单次数 > 边界值**：
      * 记录错误日志（提示在多少秒内撤单了多少次）。
      * **拉黑**：将该合约加入 `_exclude_codes`。
      * 返回 `false`。
  * **清理缓存**：为了防止内存无限增长，移除时间窗口之前的过期时间戳。
* **通过检查**
  * 如果未触发上述任何限制，返回 `true`。
```cpp
/**
 * @brief 检查撤单限制
 * @param stdCode 标准合约代码
 * @return 允许撤单返回true，禁止撤单返回false
 */
bool TraderAdapter::checkCancelLimits(const char* stdCode)
```

##### 检查下单限制 checkOrderLimits
用于**检查下单限制**。它在每次下单（开仓/平仓）前被调用，逻辑结构与 `checkCancelLimits` 高度一致，主要用于限制频繁报单（刷单）行为。
* **黑名单预检**
  * 如果合约已经在 `_exclude_codes` 中，直接返回 `false`。
* **获取风控参数**
  * 获取对应的风控配置。如果没有配置，默认允许，返回 `true`。
* **总下单次数检查**
  * 获取统计信息 `statInfo`。
  * 如果配置了总限额 (`_order_total_limits != 0`) 且 **当前累计下单次数 >= 总限额**：
    * 记录错误日志。
    * **拉黑**：将该合约加入 `_exclude_codes`。
    * 返回 `false`。
* **下单频率检查（流量控制）**
  * 获取该合约的下单时间戳缓存 `_order_time_cache`。
  * 如果缓存数量达到边界值 (`_order_times_boundary`)：
    * **计算时间窗口**：根据配置的统计时间跨度 `_order_stat_timespan` 计算起始时间 `sTime`。
    * **统计区间次数**：利用 `lower_bound` 统计该时间段内的下单数量。
    * **判定**：如果 **区间内下单次数 > 边界值**：
      * 记录错误日志。
      * **拉黑**：将该合约加入 `_exclude_codes`。
      * 返回 `false`。
    * **清理缓存**：移除过期的时间戳记录。
* **通过检查**
  * 如果未触发任何限制，返回 `true`。
```cpp
/**
 * @brief 检查下单限制
 * @param stdCode 标准合约代码
 * @return 允许下单返回true，禁止下单返回false
 */
bool TraderAdapter::checkOrderLimits(const char* stdCode)
```

#### ITraderSpi接口

##### 处理交易事件 handleEvent
用于处理交易接口的通用事件，是连接生命周期的总控制器。它主要负责响应底层的连接建立或断开事件，并自动触发后续的登录流程或断线通知。
* **连接事件 (`WTE_Connect`)**
  * **连接成功 (`ec == 0`)**：
    * 从 `_cfg` 中读取配置参数（用户名 `user`、密码 `pass`、产品 `product`）。
    * 自动调用 `_trader_api->login` 发起登录请求，进入认证流程。
  * **连接失败**：记录错误日志，提示连接失败的原因（错误码）。
* **断开事件 (`WTE_Close`)**
  * **资源清理与通知**：
    * 记录断开连接的错误日志。
    * 遍历所有注册的监听器 (`_sinks`)，调用 `on_channel_lost` 通知上层策略或引擎 *交易通道断开了*，以便触发相应的应急机制
```cpp
/**
 * @brief 处理交易事件
 * @param e 交易事件类型
 * @param ec 事件代码
 */
void TraderAdapter::handleEvent(WTSTraderEvent e, int32_t ec)
```

##### 登录结果回调 onLoginResult
登录请求的回调入口。它负责处理登录结果，并在登录成功后自动启动 *初始化查询链*（查持仓 -> 查订单 -> 查成交 -> 查账户）。
* **登录失败 (`!bSucc`)**
  * 修改状态为 `AS_LOGINFAILED`。
  * 记录错误日志，打印失败原因。
* **登录成功**
  * **状态更新**：修改状态为 `AS_LOGINED`（已登录），并保存当前的交易日 `_trading_day`。
  * **启动初始化链**：
    * 记录成功日志。
    * **第一步**：立即调用 `_trader_api->queryPositions()` 开始查询持仓。这是初始化数据同步的第一环，后续步骤会在对应的回调函数中级联触发。
```cpp
/**
 * @brief 登录结果回调
 * @param bSucc 登录是否成功
 * @param msg 消息
 * @param tradingdate 交易日
 */
void TraderAdapter::onLoginResult(bool bSucc, const char* msg, uint32_t tradingdate)
```

##### 登出回调 onLogout
```cpp
/**
 * @brief 登出回调
 * 实现ITraderSpi接口，处理登出回调。
 */
void TraderAdapter::onLogout()
{
	
}
```

##### 委托回报回调 onRspEntrust
处理委托回报（尤其是错误回报）。当委托发送到柜台或交易所被拒单、出错时，该函数会被调用。
* **错误检查**
  * 检查错误对象 `err` 是否存在且错误码不为 `WEC_NONE`。
* **回滚未完成数量**
  * **去重逻辑**：在实盘中，错误回报可能会被推送多次。函数先检查该合约当前的 `_undone_qty`。如果为 0，说明已经处理过（或原本就没有挂单），直接跳过，避免重复扣减。
  * **修正数据**：调用 `updateUndone`，传入负的数量（`-qty`），将之前下单时预加的“未完成数量”减回去，恢复到下单前的状态。
* **通知上层**
  * 解析 `UserTag` 获取本地订单 ID。
  * 遍历监听器 `_sinks`，调用 `on_entrust` 通知策略层该笔委托失败（`false`）及错误信息。
```cpp
/**
 * @brief 委托回报回调
 * @param entrust 委托单指针
 * @param err 错误信息指针
 */
void TraderAdapter::onRspEntrust(WTSEntrust* entrust, WTSError *err)
```

##### 账户查询回调 onRspAccount
账户资金查询的回调，通常也是初始化流程的**最后一步**。它标志着交易适配器已经完成了所有必要数据的同步，进入 *完全就绪* 状态。
* **状态检查**
  * 检查当前状态是否为 `AS_TRADES_QRYED`（即“成交记录已查询完毕”）。这确保了初始化流程是按顺序执行的。
* **完成初始化**
  * **状态更新**：将状态设置为 `AS_ALLREADY`（全部就绪）。
  * **就绪通知**：
    * 记录日志 *Trading channel ready*
    * 遍历监听器 `_sinks`，调用 `on_channel_ready`，正式通知系统交易通道已打通，策略可以开始交易了。
```cpp
/**
 * @brief 账户查询回调
 * @param ayAccounts 账户数组
 */
void TraderAdapter::onRspAccount(WTSArray* ayAccounts)
```

##### 持仓查询回调 onRspPosition
该函数是**持仓查询的回调**。它负责将底层的持仓数据同步到适配器内部，并根据 *初始化链* 的逻辑触发下一步查询（查订单）。
* **数据同步**
  * 遍历收到的持仓数组 `ayPositions`。
  * 获取合约信息，根据多空方向（`WDT_LONG`/`WDT_SHORT`），将底层的**今仓/昨仓**、**总持仓/可用持仓**分别更新到内部的 `_positions` 映射表中。
  * 打印日志并通知 `_sinks` 中的所有监听器 (`sink->on_position`) 更新持仓状态。
* **级联查询**
  * 如果当前状态是 `AS_LOGINED`（说明处于刚登录后的初始化阶段）：
    * **状态更新**：将状态切换为 `AS_POSITION_QRYED`（持仓已查询）。
    * **下一步**：自动调用 `_trader_api->queryOrders()`，启动**查询订单**的流程。
```cpp
/**
 * @brief 持仓查询回调
 * @param ayPositions 持仓数组
 */
void TraderAdapter::onRspPosition(const WTSArray* ayPositions)
```

##### 订单查询回调 onRspOrders
该函数是**订单查询的回调**。它负责在初始化阶段将交易所的 *在途订单* 同步到本地，重建 *未完成数量* 缓存，并更新交易统计数据。
* **数据初始化**
  * 如果内部订单映射表 `_orders` 为空，先进行创建。
  * 清空未完成数量缓存 `_undone_qty`，准备重新计算。
* **遍历与处理**
  * 遍历收到的订单数组 `ayOrders`。
  * **统计更新**：无论是不是本策略发出的单子，只要归属于该账户，都会被用来更新 `_stat_map` 中的统计数据（如挂单量、撤单量、错单量），用于风控统计。
  * **筛选与重建**：
    * 检查订单是否处于活跃状态（`isAlive`）。
    * 检查订单是否匹配当前适配器的 `_order_pattern`，只有匹配的订单才会被加入 `_orders` 管理。
    * **重建未完成量**：累加这些活跃订单的剩余数量到 `_undone_qty` 中，确保策略重启后能正确知道还有多少单子挂在外面。
* **级联查询**
  * 如果当前状态 `_state` 是 `AS_POSITION_QRYED`（持仓已查完）：
    * **状态更新**：切换为 `AS_ORDERS_QRYED`。
    * **下一步**：自动调用 `_trader_api->queryTrades()`，启动**查询成交**的流程。
```cpp
/**
 * @brief 订单查询回调
 * @param ayOrders 订单数组
 */
void TraderAdapter::onRspOrders(const WTSArray* ayOrders)
```

##### 成交查询回调 onRspTrades
该函数是**成交查询的回调**。它主要用于在初始化阶段同步当天的成交记录，从而正确初始化交易统计信息。
* **统计重建**
  * 遍历收到的成交数组 `ayTrades`。
  * 根据成交的方向（多/空）和开平类型（开/平/平今），累加到 `_stat_map[stdCode]` 中。
    * 例如：如果是 *买入开仓*，则增加 `l_openvol`（多头开仓量）。
  * 这些统计数据对于后续的 `checkOrderLimits` 等风控检查至关重要。
* **级联查询**
  * 如果当前状态是 `AS_ORDERS_QRYED`（订单已查完）：
    * **状态更新**：切换为 `AS_TRADES_QRYED`。
    * **下一步**：自动调用 `_trader_api->queryAccount()`，启动**查询资金**的流程（这是初始化链的最后一环）。
```cpp
/**
 * @brief 成交查询回调
 * @param ayTrades 成交数组
 */
void TraderAdapter::onRspTrades(const WTSArray* ayTrades)
```

##### 订单推送回调 onPushOrder
该函数是 **订单状态推送（回报）的处理核心**。它不仅仅是简单地转发订单状态，还承担了**风控统计**、**未完成单管理**、**可用持仓冻结与解冻**以及**内部订单生命周期维护**等关键职责。

* **前置校验与信息获取**
  * **空指针检查**：首先检查传入的 `orderInfo` 是否为空。
  * **获取合约信息**：通过 `_bd_mgr` 获取合约信息 `cInfo`，如果获取失败则直接返回。这是后续获取标准合约代码（`stdCode`）的基础。
  * **判断交易方向**：根据订单的方向和开平标志，判断该订单是 **买入行为**（开多或平空）还是 **卖出行为**（开空或平多），主要用于后续的统计分类。
* **交易统计更新**
  * 该模块用于维护 `_stat_map` 中的统计数据，这些数据是**风控模块**判断是否“频繁撤单”或“错单过多”的依据。
  * **获取统计对象**：尝试从 `_stat_map` 获取该合约的统计信息，如果不存在则创建并添加。
  * **处理撤单/错单统计**：只有当订单状态为 **已撤销 (`WOS_Canceled`)** 时才更新：
    * **错单统计**：如果 `isError()` 为真，增加错单次数 (`wrongs`) 和错单量。
    * **撤单统计**：如果不是错单，区分普通撤单和自动撤单（FAK/FOK）：
      * **普通单 (`WOF_NOR`)**：增加 `cancels` 计数。
      * **自动单**：增加 `auto_cancels` 计数。
    * 上述统计均根据“买入”或“卖出”方向分别记录。
* **未完成数量维护**
  * 该模块确保策略引擎能准确知道当前有多少“挂在外面”的单子，防止策略重复发单。
  * **条件**：订单状态为 **已撤销** 且 **用户标签 (`UserTag`)** 符合本适配器的命名模式（说明是自己发出的单）。
  * **逻辑**：
    * 计算撤销的剩余数量：`qty = 总量 - 已成交量`。
    * 调用 `updateUndone`，传入负数（`-qty`），从而减少该合约的未完成数量记录。
    * 打印详细的撤单日志。
* **可用持仓的冻结与解冻**
  * 这是该函数最复杂也最重要的逻辑，用于保证持仓数据的准确性，防止超仓卖出。
  * **新订单冻结**
    * 当一个订单**第一次**被推送过来时（通过 `_orderids` 集合判断是否已处理过）：
    * **记录 ID**：将订单号加入 `_orderids` 防止重复处理。
    * **增加信息量**：统计项 `_infos` 加 1。
    * **冻结逻辑**：只有 **平仓订单**（`OffsetType != WOT_OPEN`）才需要冻结可用持仓：
      * **平今 (`WOT_CLOSETODAY`)**：直接扣减 **今仓可用** (`newavail`)。
      * **普通平仓 (`WOT_CLOSE`)**：
        1. 优先扣减 **昨仓可用** (`preavail`)。
        2. 如果昨仓不够扣，剩余部分扣减 **今仓可用** (`newavail`)。
      * **注意**：这里使用了 `min` 函数，防止扣减过量变成负数。
  * **撤单解冻**
    * 当一个 **非第一次推送** 且 **已撤销** 的 **平仓订单** 到达时，说明之前的冻结操作需要回滚：
    * **恢复逻辑**：
      * **平今 (`WOT_CLOSETODAY`)**：直接将撤销量加回 **今仓可用** (`newavail`)。
      * **普通平仓 (`WOT_CLOSE`)**：
        1. 先尝试加回 **昨仓可用** (`preavail`)。
        2. **溢出处理**：如果加回后的昨仓可用量超过了实际昨仓总量（`prevol`），说明当初冻结时借用了今仓的额度。因此，将溢出的部分加回 **今仓可用** (`newavail`)，并将昨仓可用修正为最大值（即昨仓总量）。
* **内部订单管理与通知**（如果该单子由本适配器发出去，即订单标签与 `_order_pattern` 一致）
  * 该模块负责维护 `_orders` 映射表，保持与交易所状态一致，并通知上层策略。
  * **解析本地 ID**：检查 `UserTag` 是否以 `wtp.` 开头，解析出 `localid`。
  * **更新订单表**：
    * **加锁**：使用 `_mtx_orders` 自旋锁保证线程安全。
    * **移除**：如果订单已结束（`!isAlive`，如全成、撤销、拒单），从 `_orders` 中移除，释放内存。
    * **更新/添加**：如果订单仍活跃（如部分成交、挂单中），更新或添加到 `_orders` 中。
  * **回调通知**：
    * 转换开平标志（0:开, 1:平, 2:平今）。
    * 遍历所有注册的 `sink`，调用 `on_order` 将订单的最新状态（ID、代码、方向、开平、剩余量、价格、是否撤单）推送给策略或引擎。
```cpp
/**
 * @brief 订单推送回调
 * @param orderInfo 订单信息指针
 */
void TraderAdapter::onPushOrder(WTSOrderInfo* orderInfo)
```

##### 成交推送回调 onPushTrade
该函数是 **成交回报的处理核心**。每当交易所撮合了一笔交易，都会推送一条成交记录。此函数负责根据这条记录，**实时**更新本地的持仓数据、扣减挂单（未完成）数量，并将成交信息通知给策略，同时触发资金查询以保持账户信息同步。
* **基础信息准备与校验**
  * **获取合约与品种信息**：
    * 通过基础数据管理器 `_bd_mgr` 获取合约信息 `cInfo`，确保能拿到标准合约代码（`stdCode`）。
    * 获取品种信息 `commInfo`，这一步非常关键，因为后续需要用到品种的 **T+1 属性**（`isT1()`）来判断开仓后是否立即增加可用持仓。
  * **状态标志判断**：
    * `isLong`：判断成交方向是否为多头（`WDT_LONG`）。
    * `isOpen`：判断开平标志是否为开仓（`WOT_OPEN`）。
* **未完成数量维护**
  * 该模块用于维护 *挂在外面等待成交的单子数量*。成交意味着挂单变成了持仓，因此必须减少未完成数量。
  * **自身订单识别**：
    * 检查成交记录的 `UserTag` 是否以本适配器的前缀（`_order_pattern`）开头。只有自己的单子才需要更新本地的未完成计数。
  * **扣减逻辑**：
    * 解析出本地订单 ID (`localid`)。
    * 调用 `updateUndone(stdCode, -vol)`，传入负数，表示减少该合约的挂单数量。这是保证策略不重复发单的重要依据。
* **持仓实时更新**
  * 这是函数中最核心的逻辑。它直接修改内存中的持仓结构 `PosItem`
  * **统计对象准备**
    * 获取或创建该合约的交易统计对象 `WTSTradeStateInfo`（虽然代码中获取了它，但在此函数片段中未见显式修改统计值，通常用于确保映射表中有该合约的条目）。
  * **多头持仓更新** (`isLong == true`)
    * **开仓 (`isOpen`)**：
      * **增加持仓**：`l_newvol`（今仓总量）增加成交量。
      * **增加可用**：检查 `!commInfo->isT1()`。
        * 如果是期货（非 T+1），开仓即平，`l_newavail`（今仓可用）立即增加。
        * 如果是股票（T+1），开仓当天不可平，因此不增加可用量。
    * **平今 (`WOT_CLOSETODAY`)**：
      * 直接扣减 `l_newvol`（今仓总量）。注意：可用量（`newavail`）在发单冻结时已经扣除，此处只需更新总量。
    * **普通平仓 (`WOT_CLOSE`)**：
      * **优先平昨**：先尝试从 `l_prevol`（昨仓总量）中扣除成交量。
      * **余额平今**：如果昨仓不够（例如昨仓 5 手，平仓 8 手），剩下的 3 手从 `l_newvol`（今仓总量）中扣除。
      * 这符合大多数交易所“先开先平”或“优先平昨”的默认撮合规则。
  * **空头持仓更新** (`isLong == false`)
    * 逻辑与多头完全对称：
    * **开仓**：增加 `s_newvol`。期货通常非 T+1，所以也增加 `s_newavail`。
    * **平今**：减少 `s_newvol`。
    * **普通平仓**：优先减少 `s_prevol`，不够再减少 `s_newvol`。
* **日志与调试**
  * 打印详细的调试日志，包含合约代码、`UserTag`、成交量、成交价等关键信息，便于追溯。
  * 调用 `printPosition` 打印更新后的持仓快照，方便核对逻辑是否正确。
* **通知与资金同步**
  * **回调通知**：
    * 将底层的开平类型转换为统一的整数标志（0:开, 1:平, 2:平今）。
    * 遍历所有注册的监听器 (`_sinks`)，调用 `on_trade`，将成交细节推送给上层策略引擎。
  * **触发资金查询**：
    * 最后调用 `_trader_api->queryAccount()`。
    * **目的**：成交会产生手续费，并释放或占用保证金。为了让策略能获取最准确的资金余额（用于风控或仓位计算），必须在成交后立即刷新账户信息。
```cpp
/**
 * @brief 成交推送回调
 * @param tradeRecord 成交记录指针
 */
void TraderAdapter::onPushTrade(WTSTradeInfo* tradeRecord)
```

##### 交易错误回调 onTraderError
```cpp
/**
 * @brief 交易错误回调
 * @param err 错误信息指针
 * @param pData 附加数据指针
 * 实现ITraderSpi接口，处理交易错误。
 */
void TraderAdapter::onTraderError(WTSError* err, void* pData /* = NULL */)
{
	if(err)
		WTSLogger::log_dyn("trader", _id.c_str(), LL_ERROR,"[{}] Error of trading channel occured: {}", _id.c_str(), err->getMessage());
}
```

##### 获取基础数据管理器 getBaseDataMgr
```cpp
/**
 * @brief 获取基础数据管理器
 * @return 基础数据管理器指针
 */
IBaseDataMgr* TraderAdapter::getBaseDataMgr()
{
	return _bd_mgr;
}
```

##### 处理交易日志 handleTraderLog
```cpp
/**
 * @brief 处理交易日志
 * @param ll 日志级别
 * @param message 日志消息
 */
void TraderAdapter::handleTraderLog(WTSLogLevel ll, const char* message)
{
	WTSLogger::log_dyn_raw("trader", _id.c_str(), ll, message);
}
```

#### 内部方法

##### 执行委托下单 doEntrust
该函数用于**执行底层的委托下单逻辑**。它是所有上层下单接口（如 `openLong`, `closeShort` 等）的公共入口，负责完善委托单信息（ID、交易所、Tag 等）并调用具体的交易 API 发送指令：

* **生成委托单 ID**
  * 调用交易接口 API  `_trader_api` 生成唯一的委托单 ID（字符串形式），并填充到委托对象 `entrust` 中。
* **解析与设置合约代码**
  * 传入的合约代码通常是标准代码格式（如 `SHFE.rb2305`）。函数需要将其拆分为交易所代码和合约代码，并分别设置到委托对象 `entrust` 中
* **生成本地订单 ID 与 UserTag**
  * 为了在收到回报时能识别是我们发出的订单，需要生成唯一的本地标识：
    * **生成 ID**：调用 `makeLocalOrderID()` 生成一个自增的 `uint32_t` 本地订单 ID。
    * **构造 Tag**：将 ID 格式化为特定的字符串模式（如 `wtp.[适配器ID].[本地ID]`），并写入委托单的 `UserTag` 字段。这个 Tag 是后续关联回报与本地订单的关键。
* **调用 API 发送指令**
  * 调用交易接口 `_trader_api` 的 `orderInsert(entrust)` 发送委托。
  * **错误处理**：如果返回值小于 0，记录错误日志，并返回 `UINT_MAX` 表示失败。
  * **成功返回**：如果发送成功，返回生成的 `localid`。
```cpp
/**
 * @brief 执行委托下单
 * @param entrust 委托单指针
 * @return 本地订单ID，失败返回UINT_MAX
 * * 内部方法，处理委托单的下单逻辑，生成本地订单ID。
 */
uint32_t TraderAdapter::doEntrust(WTSEntrust* entrust)
```

##### 执行撤单 doCancel
执行撤单操作的内部核心实现，无论是按 ID 撤单 (`cancel`) 还是全部撤单 (`cancelAll`)，最终都会调用此函数来构造请求并发送给交易接口。
* **前置校验**
  * **有效性检查**：判断传入的订单信息 `ordInfo` 是否为空，或者订单是否已经结束（`!isAlive()`）。如果是，则直接返回 `false`，因为死单无法撤销。
  * **获取合约信息**：尝试从订单信息中获取 `WTSContractInfo`。如果缺失，则通过基础数据管理器 `_bd_mgr` 重新查询。
* **构造撤单请求**
  * **创建动作对象**：创建一个 `WTSEntrustAction` 对象，传入合约代码和交易所代码。
  * **填充 ID**：将订单的 `EntrustID`（委托编号）和 `OrderID`（报单编号）填入动作对象，以便交易所识别要撤销哪一笔订单。
* **调用接口与清理**
  * **发送请求**：调用底层 API `_trader_api->orderAction(action)` 发送撤单指令。
  * **结果判定**：检查返回值 `ret`，如果不小于 0 则视为发送成功 (`isSent = true`)。
  * **释放资源**：调用 `action->release()` 释放刚才创建的动作对象内存。
  * **返回**：返回发送结果。
```cpp
/**
 * @brief 执行撤单
 * @param ordInfo 订单信息指针
 * @return 撤单成功返回true，失败返回false
 * * 内部方法，处理订单的撤单逻辑。
 */
bool TraderAdapter::doCancel(WTSOrderInfo* ordInfo)
```

##### 获取合约信息 getContract
```cpp
/**
 * @brief 根据标准代码获取合约信息
 * @param stdCode 标准合约代码
 * @return 合约信息指针，失败返回NULL
 */
WTSContractInfo* TraderAdapter::getContract(const char* stdCode)
{
	CodeHelper::CodeInfo cInfo = CodeHelper::extractStdCode(stdCode, NULL);
	return _bd_mgr->getContract(cInfo._code, cInfo._exchg);
}
```

##### 更新未完成数量 updateUndone
将 qty 加到 `_undone_qty[stdCode]` 上
```cpp
/**
 * @brief 更新未完成数量
 * @param stdCode 合约代码
 * @param qty 数量变化（正数表示增加，负数表示减少）
 */
void TraderAdapter::updateUndone(const char* stdCode, double qty)
```

##### 获取风险控制参数 getRiskParams
获取指定合约的风控参数：
* **解析品种代码**
  * 输入的 `stdCode` 通常是标准格式（如 `SHFE.cu2312`）。
  * **定位分隔符**：找到第一个 `.` 的位置。
  * **提取品种**：从分隔符后开始遍历，提取连续的**字母**部分（例如从 `cu2312` 中提取出 `cu`），忽略后面的数字年份和月份。
* **查找风控规则**
  * **特定品种匹配**：使用提取出的品种代码（如 `cu`）在风控参数映射表 `_risk_params_map` 中查找。如果找到，直接返回该品种的配置指针。
  * **默认规则匹配**：如果未找到特定品种的配置，则查找键名为 `"default"` 的默认配置规则。
* **返回结果**
  * 如果特定品种和默认规则都未找到，则返回 `NULL`，表示该合约不受风控限制（或配置缺失）。
```cpp
/**
 * @brief 获取风险控制参数
 * @param stdCode 标准合约代码
 * @return 风险控制参数指针，如果未找到则返回默认参数（如果未找到默认参数则可能返回空指针）
 */
const TraderAdapter::RiskParams* TraderAdapter::getRiskParams(const char* stdCode)
```

### 交易适配器管理器类 TraderAdapterMgr
```cpp
class TraderAdapterMgr : private boost::noncopyable
```

#### 成员
- `TraderAdapterMap _adapters`：交易适配器映射表，键为交易通道名称
  - `typedef wt_hashmap<std::string, TraderAdapterPtr> TraderAdapterMap`：交易适配器映射表类型
  - `typedef std::shared_ptr<TraderAdapter> TraderAdapterPtr`：交易适配器智能指针类型

#### 添加适配器 addAdapter
```cpp
/**
 * @brief 添加交易适配器
 * @param tname 交易通道名称
 * @param adapter 交易适配器智能指针
 * @return 添加成功返回true，失败返回false
 * 将交易适配器添加到管理器中，如果名称已存在则添加失败。
 */
bool TraderAdapterMgr::addAdapter(const char* tname, TraderAdapterPtr& adapter)
{
	if (adapter == NULL || strlen(tname) == 0)
		return false;  // 返回失败
	auto it = _adapters.find(tname);
	if(it != _adapters.end())
	{
		WTSLogger::error("Same name of trading channels: {}", tname);
		return false;
	}

	_adapters[tname] = adapter;
	return true;
}
```

#### 获取指定名称的适配器 getAdapter
```cpp
/**
 * @brief 获取指定名称的适配器
 * @param tname 交易通道名称
 * @return 交易适配器智能指针，如果不存在则返回空指针
 */
TraderAdapterPtr TraderAdapterMgr::getAdapter(const char* tname)
{
	auto it = _adapters.find(tname);
	if (it != _adapters.end())
	{
		return it->second;
	}
	return TraderAdapterPtr();
}
```

#### 获取适配器映射表 getAdapters
```cpp
/**
 * @brief 获取适配器映射表
 * @return 适配器映射表常量引用
 */
const TraderAdapterMap& getAdapters() const { return _adapters; }
```

#### 启动所有适配器 run
```cpp
/**
 * @brief 启动所有适配器
 * 启动所有交易适配器，开始连接和登录流程。
 */
void TraderAdapterMgr::run()
{
	for (auto it = _adapters.begin(); it != _adapters.end(); it++)
	{
		it->second->run();
	}
	WTSLogger::info("{} trading channels started", _adapters.size());
```

# 策略管理层

## 框架图
```mermaid
graph LR
    %% 样式定义
    classDef dataStruct fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef dataBlock fill:#bbdefb,stroke:#0d47a1,stroke-width:2px,color:#000;
    classDef context fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef wrapper fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef manager fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef interface fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef strategy fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#000;

    %% =======================
    %% 数据定义层 - UftDataDefs.h
    %% =======================
    subgraph DataLayer["数据定义层 - UftDataDefs.h"]
        direction TB
        
        subgraph BaseStruct["基础结构"]
            BlockHeader["BlockHeader<br/>数据块头<br/>标志/类型/日期/容量/大小"]:::dataStruct
        end
        
        subgraph DataStructs["数据结构"]
            DetailStruct["DetailStruct<br/>持仓明细结构<br/>交易所/合约/方向/数量/价格/盈亏"]:::dataStruct
            OrderStruct["OrderStruct<br/>订单结构<br/>交易所/合约/方向/开平/数量/价格/状态"]:::dataStruct
            TradeStruct["TradeStruct<br/>成交结构<br/>交易所/合约/方向/开平/数量/价格/时间"]:::dataStruct
            RoundStruct["RoundStruct<br/>回合结构<br/>交易所/合约/方向/开平价格/盈亏"]:::dataStruct
        end
        
        subgraph DataBlocks["数据块"]
            PositionBlock["PositionBlock<br/>持仓数据块<br/>继承BlockHeader<br/>包含DetailStruct数组"]:::dataBlock
            OrderBlock["OrderBlock<br/>订单数据块<br/>继承BlockHeader<br/>包含OrderStruct数组"]:::dataBlock
            TradeBlock["TradeBlock<br/>成交数据块<br/>继承BlockHeader<br/>包含TradeStruct数组"]:::dataBlock
            RoundBlock["RoundBlock<br/>回合数据块<br/>继承BlockHeader<br/>包含RoundStruct数组"]:::dataBlock
        end
    end

    %% =======================
    %% 策略上下文层 - UftStraContext.h/cpp
    %% =======================
    subgraph ContextLayer["策略上下文层 - UftStraContext"]
        direction TB
        
        subgraph ContextClass["上下文类"]
            UftStraContext["UftStraContext<br/>策略上下文<br/>实现IUftStraCtx和ITrdNotifySink<br/>管理持仓/订单/成交/回合<br/>数据持久化与事件转发"]:::context
        end
        
        subgraph BlockPairs["数据块配对"]
            PosBlkPair["PosBlkPair<br/>持仓数据块配对<br/>PositionBlock + 内存映射文件 + 锁"]:::dataBlock
            OrdBlkPair["OrdBlkPair<br/>订单数据块配对<br/>OrderBlock + 内存映射文件 + 锁"]:::dataBlock
            TrdBlkPair["TrdBlkPair<br/>成交数据块配对<br/>TradeBlock + 内存映射文件 + 锁"]:::dataBlock
            RndBlkPair["RndBlkPair<br/>回合数据块配对<br/>RoundBlock + 内存映射文件 + 锁"]:::dataBlock
        end
    end

    %% =======================
    %% 策略管理层 - UftStrategyMgr.h/cpp
    %% =======================
    subgraph ManagerLayer["策略管理层 - UftStrategyMgr"]
        direction TB
        
        subgraph WrapperClass["包装器类"]
            UftStraWrapper["UftStraWrapper<br/>策略包装器<br/>包装策略实例和工厂<br/>管理策略生命周期"]:::wrapper
        end
        
        subgraph ManagerClass["管理器类"]
            UftStrategyMgr["UftStrategyMgr<br/>策略管理器<br/>加载策略工厂<br/>创建和管理策略实例"]:::manager
        end
    end

    %% =======================
    %% 外部接口层
    %% =======================
    subgraph InterfaceLayer["外部接口层"]
        direction TB
        IUftStraCtx["IUftStraCtx<br/>策略上下文接口<br/>交易接口/数据查询/持仓管理"]:::interface
        ITrdNotifySink["ITrdNotifySink<br/>交易通知接口<br/>成交回报/订单回报/持仓更新"]:::interface
        UftStrategy["UftStrategy<br/>策略基类<br/>定义策略生命周期回调"]:::strategy
        IUftStrategyFact["IUftStrategyFact<br/>策略工厂接口<br/>创建/删除策略实例"]:::interface
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    PositionBlock -.->|"继承"| BlockHeader
    OrderBlock -.->|"继承"| BlockHeader
    TradeBlock -.->|"继承"| BlockHeader
    RoundBlock -.->|"继承"| BlockHeader
    
    UftStraContext -.->|"实现"| IUftStraCtx
    UftStraContext -.->|"实现"| ITrdNotifySink
    UftStrategy -.->|"由工厂创建"| IUftStrategyFact

    %% =======================
    %% 组合关系（实线）
    %% =======================
    PositionBlock -->|"包含"| DetailStruct
    OrderBlock -->|"包含"| OrderStruct
    TradeBlock -->|"包含"| TradeStruct
    RoundBlock -->|"包含"| RoundStruct
    
    PosBlkPair -->|"使用"| PositionBlock
    OrdBlkPair -->|"使用"| OrderBlock
    TrdBlkPair -->|"使用"| TradeBlock
    RndBlkPair -->|"使用"| RoundBlock
    
    UftStraContext -->|"管理"| PosBlkPair
    UftStraContext -->|"管理"| OrdBlkPair
    UftStraContext -->|"管理"| TrdBlkPair
    UftStraContext -->|"管理"| RndBlkPair
    UftStraContext -->|"绑定"| UftStrategy
    
    UftStraWrapper -->|"包装"| UftStrategy
    UftStraWrapper -->|"持有"| IUftStrategyFact
    UftStrategyMgr -->|"管理"| UftStraWrapper
    UftStrategyMgr -->|"加载"| IUftStrategyFact

    %% =======================
    %% 数据流关系
    %% =======================
    UftStrategy -->|"使用"| UftStraContext
    UftStraContext -->|"事件转发"| UftStrategy
    UftStrategyMgr -->|"创建策略"| UftStraWrapper

    %% 应用样式
    class BlockHeader,DetailStruct,OrderStruct,TradeStruct,RoundStruct dataStruct
    class PositionBlock,OrderBlock,TradeBlock,RoundBlock,PosBlkPair,OrdBlkPair,TrdBlkPair,RndBlkPair dataBlock
    class UftStraContext context
    class UftStraWrapper wrapper
    class UftStrategyMgr manager
    class IUftStraCtx,ITrdNotifySink,IUftStrategyFact interface
    class UftStrategy strategy
```

## UftDataDefs.h — 数据定义

## UftStraContext.h/cpp — 策略上下文

### UFT策略上下文类 UftStraContext
```cpp
class UftStraContext : public IUftStraCtx, public ITrdNotifySink
```

#### 成员
- **核心标识与指针**
  - `uint32_t _context_id`：上下文ID，策略上下文的唯一标识
  - `WtUftEngine* _engine`：UFT引擎指针
  - `TraderAdapter* _trader`：交易适配器指针
  - `UftStrategy* _strategy`：策略对象指针
  - `uint32_t _tradingday`：当前交易日

- **本地数据块配对对象**
  - `PosBlkPair _pos_blk`：持仓数据块配对对象
    ```cpp
    /**
     * @struct PosBlkPair
    * @brief 持仓数据块配对结构体
    * 用于管理持仓数据的内存映射文件和互斥锁。
    */
    typedef struct _PosBlkPair
    {
      uft::PositionBlock* _block; // 持仓数据块指针
      BoostMFPtr _file; // 内存映射文件智能指针
      SpinMutex _mutex; // 自旋互斥锁，用于线程安全
    } PosBlkPair;
    ```
  - `OrdBlkPair _ord_blk`：订单数据块配对对象
    ```cpp
    /**
    * @struct OrdBlkPair
    * @brief 订单数据块配对结构体
    * 用于管理订单数据的内存映射文件和互斥锁。
    */
    typedef struct _PosBlkPair
    {
      uft::OrderBlock* _block; // 订单数据块指针
      BoostMFPtr _file; // 内存映射文件智能指针
      SpinMutex _mutex; // 自旋互斥锁，用于线程安全
    } PosBlkPair;
    ```
  - `TrdBlkPair _trd_blk`：成交数据块配对对象
    ```cpp
    /**
     * @struct TrdBlkPair
    * @brief 成交数据块配对结构体
    * 用于管理成交数据的内存映射文件和互斥锁。
    */
    typedef struct _TrdBlkPair
    {
      uft::TradeBlock* _block; // 成交数据块指针
      BoostMFPtr _file; // 内存映射文件智能指针
      SpinMutex _mutex; // 自旋互斥锁，用于线程安全
    } TrdBlkPair;
    ```
  - `RndBlkPair _rnd_blk`：回合数据块配对对象
    ```cpp
    /**
     * @struct RndBlkPair
    * @brief 回合数据块配对结构体
    * 用于管理回合数据的内存映射文件和互斥锁。
    */
    typedef struct _RndBlkPair
    {
      uft::RoundBlock* _block; // 回合数据块指针
      BoostMFPtr _file; // 内存映射文件智能指针
      SpinMutex _mutex; // 自旋互斥锁，用于线程安全
    } RndBlkPair;
    ```

- **持仓与订单映射表**
  - `wt_hashmap<std::string, PosInfo> _positions`：持仓信息映射表
    - key为合约代码（std::string），value为PosInfo结构体
      ```cpp
        /**
         * @struct PosInfo
        * @brief 持仓信息结构体
        * 用于管理单个合约的持仓信息，包括持仓数量、开仓成本、盈亏等。
        */
        typedef struct _Position
        {
          // 多仓数据（净持仓，正数表示多仓，负数表示空仓）
          double _volume; // 持仓数量（净持仓）
          double _opencost; // 开仓成本
          double _dynprofit; // 动态盈亏（持仓盈亏）
          double _total_profit; // 总盈亏（持仓盈亏+平仓盈亏）
          uint32_t _valid_idx; // 有效索引，用于跳过已平仓的持仓明细
          std::vector<uft::DetailStruct*> _details; // 持仓明细列表
        } PosInfo;
      ```
  - `wt_hashmap<uint32_t, uft::OrderStruct*> _order_ids`：订单ID映射表
    - key为本地订单ID（uint32_t）
    - value为订单结构体指针（uft::OrderStruct*）

#### 策略对象管理

##### 设置策略对象 set_strategy
```cpp
/**
 * @brief 设置策略对象
 * @param stra 策略对象指针
 * 将策略对象绑定到上下文。
 */
void set_strategy(UftStrategy* stra){ _strategy = stra; }
```

##### 获取策略对象 get_stragety
```cpp
/**
 * @brief 获取策略对象
 * @return 策略对象指针
 * 返回当前绑定的策略对象。
 */
UftStrategy* get_stragety() { return _strategy; }
```

##### 设置交易适配器 setTrader
```cpp
/**
 * @brief 设置交易适配器
 * @param trader 交易适配器指针
 * 将交易适配器绑定到上下文，用于执行交易操作。
 */
void setTrader(TraderAdapter* trader);
```

#### IUftStraCtx—策略生命周期与事件回调

##### 初始化回调 on_init
```cpp
/**
 * @brief 初始化回调实现
 * 当策略初始化时调用，通知策略进行初始化操作。
 */
void UftStraContext::on_init()
{
	if (_strategy)
		_strategy->on_init(this);
}
```

##### 交易会话开始回调 on_session_begin
```cpp
/**
 * @brief 交易会话开始回调实现
 * @param uTDate 交易日
 * 当交易会话开始时调用，通知策略新交易日开始。
 */
void UftStraContext::on_session_begin(uint32_t uTDate)
{
	if (_strategy)
		_strategy->on_session_begin(this, uTDate);
}
```

##### 交易会话结束回调 on_session_end
```cpp
/**
 * @brief 交易会话结束回调实现
 * @param uTDate 交易日
 * 当交易会话结束时调用，通知策略交易日结束。
 */
void UftStraContext::on_session_end(uint32_t uTDate)
{
	if (_strategy)
		_strategy->on_session_end(this, uTDate);
}
```

##### 参数更新回调 on_params_updated
```cpp
/**
 * @brief 参数更新回调实现
 * 当策略参数更新时调用，通知策略参数已更新。
 */
void UftStraContext::on_params_updated()
{
	if (_strategy)
		_strategy->on_params_updated();
}
```

#### IUftStraCtx—市场数据回调

##### Tick数据回调 on_tick
该函数的主要作用是**响应最新的 Tick（行情）数据**。它不仅负责将行情数据推送给用户策略，更关键的是在此刻利用最新价格**实时更新本地持仓的浮动盈亏**。
1. **查找持仓信息**
   * 根据传入的合约代码 `stdCode`，在本地持仓映射表 `_positions` 中查找是否存在该合约的持仓。
   * 如果不存在持仓，则跳过盈亏计算步骤，直接进入事件转发。
2. **更新持仓盈亏 (如果有持仓)**
   * **获取基础信息**：获取合约的**数量乘数** (`volscale`)，用于计算合约价值。
   * **更新明细盈亏**：
     * 遍历该合约下所有有效的持仓明细 (`_details`)。
     * 对每一笔明细计算持仓盈亏 (`_position_profit`)：
     * 公式：`(最新价 - 开仓价) * 持仓量 * 乘数 * 方向系数`
     * 方向系数：多仓为 1，空仓为 -1。
   * **更新总持仓动态盈亏**：
     * 根据净持仓量 (`_volume`) 的正负判断多空方向。
     * **多头持仓** (`_volume > 0`)：
       * `动态盈亏 = (最新价 * 持仓量 * 乘数) - 开仓成本`
     * **空头持仓** (`_volume < 0`)：
       * `动态盈亏 = (最新价 * 持仓量 * 乘数) + 开仓成本`
     * 注：此处空头持仓量为负数，因此 `最新价 * 负持仓量` 代表负的市场价值，加上正的开仓成本，等同于 `开仓价值 - 当前价值`。
3. **转发事件给策略**
   * 检查策略对象指针 `_strategy` 是否有效。
   * 如果有效，调用策略的 `on_tick` 回调函数，将 `this` 指针（上下文）、合约代码和最新的 Tick 数据传递给策略逻辑进行处理。
```cpp
/**
 * @brief Tick数据回调实现
 * @param stdCode 标准化合约代码
 * @param newTick 新的Tick数据
 * * 当收到新的Tick数据时调用，更新持仓盈亏并通知策略。
 * 计算每个持仓明细的持仓盈亏，以及总持仓的动态盈亏。
 */
void UftStraContext::on_tick(const char* stdCode, WTSTickData* newTick)
```

##### 订单队列数据回调 on_order_queue
```cpp
/**
 * @brief 订单队列数据回调实现
 * @param stdCode 标准化合约代码
 * @param newOrdQue 新的订单队列数据
 * 当收到新的订单队列数据时调用，转发给策略处理。
 */
void UftStraContext::on_order_queue(const char* stdCode, WTSOrdQueData* newOrdQue)
{
	if (_strategy)
		_strategy->on_order_queue(this, stdCode, newOrdQue);
}
```

##### 订单明细数据回调 on_order_detail
```cpp
/**
 * @brief 订单明细数据回调实现
 * @param stdCode 标准化合约代码
 * @param newOrdDtl 新的订单明细数据
 * 当收到新的订单明细数据时调用，转发给策略处理。
 */
void UftStraContext::on_order_detail(const char* stdCode, WTSOrdDtlData* newOrdDtl)
{
	if (_strategy)
		_strategy->on_order_detail(this, stdCode, newOrdDtl);
}
```

##### 成交明细数据回调 on_transaction
```cpp
/**
 * @brief 成交明细数据回调实现
 * @param stdCode 标准化合约代码
 * @param newTrans 新的成交明细数据
 * 当收到新的成交明细数据时调用，转发给策略处理。
 */
void UftStraContext::on_transaction(const char* stdCode, WTSTransData* newTrans)
{
	if (_strategy)
		_strategy->on_transaction(this, stdCode, newTrans);
}
```

##### K线数据回调 on_bar
```cpp
/**
 * @brief K线数据回调实现
 * @param code 合约代码
 * @param period 周期字符串
 * @param times 周期倍数
 * @param newBar 新的K线数据
 * 当收到新的K线数据时调用，转发给策略处理。
 */
void UftStraContext::on_bar(const char* code, const char* period, uint32_t times, WTSBarStruct* newBar)
{
	if (_strategy)
		_strategy->on_bar(this, code, period, times, newBar);
}
```

#### IUftStraCtx—参数监控接口

##### 监控字符串参数 watch_param

##### 监控浮点数参数 watch_param

##### 监控无符号32位整数参数 watch_param

##### 监控无符号64位整数参数 watch_param

##### 监控有符号32位整数参数 watch_param

##### 监控有符号64位整数参数 watch_param

##### 提交参数监控 commit_param_watcher

#### IUftStraCtx—参数读取接口

##### 读取字符串参数 read_param

##### 读取浮点数参数 read_param

##### 读取无符号32位整数参数 read_param

##### 读取无符号64位整数参数 read_param

##### 读取有符号32位整数参数 read_param

##### 读取有符号64位整数参数 read_param

#### IUftStraCtx—参数同步接口

##### 同步字符串参数 sync_param

##### 同步浮点数参数 sync_param

##### 同步无符号32位整数参数 sync_param

##### 同步无符号64位整数参数 sync_param

##### 同步有符号32位整数参数 sync_param

##### 同步有符号64位整数参数 sync_param

#### IUftStraCtx—时间与日期接口

##### 获取当前日期 stra_get_date

##### 获取当前时间 stra_get_time

##### 获取当前秒数 stra_get_secs

#### IUftStraCtx—交易接口：下单

##### 买入接口 stra_buy

##### 卖出接口 stra_sell

##### 开多接口 stra_enter_long

##### 开空接口 stra_enter_short

##### 平多接口 stra_exit_long

##### 平空接口 stra_exit_short

#### IUftStraCtx—交易接口：撤单

##### 撤销订单 stra_cancel

##### 撤销所有订单 stra_cancel_all

#### IUftStraCtx—数据获取接口

##### 获取商品信息 stra_get_comminfo

##### 获取K线数据 stra_get_bars

##### 获取Tick数据 stra_get_ticks

##### 获取订单明细数据 stra_get_order_detail

##### 获取订单队列数据 stra_get_order_queue

##### 获取成交明细数据 stra_get_transaction

##### 获取最新Tick数据 stra_get_last_tick

#### IUftStraCtx—持仓查询接口

##### 获取账户持仓 stra_get_position

##### 获取本地持仓 stra_get_local_position

##### 获取本地持仓盈亏 stra_get_local_posprofit

##### 获取本地平仓盈 stra_get_local_closeprofit

##### 枚举持仓 stra_enum_position

#### IUftStraCtx—市场信息查询接口

##### 获取当前价格 stra_get_price

##### 获取未完成数量 stra_get_undone

##### 获取信息数量 stra_get_infos

#### IUftStraCtx—数据订阅接口

##### 订阅Tick数据 stra_sub_ticks

##### 订阅订单明细数据 stra_sub_order_details

##### 订阅订单队列数据 stra_sub_order_queues

##### 订阅成交明细数据 stra_sub_transactions

#### IUftStraCtx—日志接口

##### 记录信息日志 stra_log_info

##### 记录调试日志 stra_log_debug

##### 记录错误日志 stra_log_error

#### ITrdNotifySink—交易回报回调 

##### 成交回报回调 on_trade

##### 订单回报回调 on_order

##### 下单回报回调 on_entrust

##### 持仓更新回调 on_position

##### 交易通道就绪回调 on_channel_ready

##### 交易通道丢失回调 on_channel_lost

#### 私有辅助方法

##### 加载本地数据 load_local_data

##### 判断是否为我的订单 is_my_order

##### 记录调试日志（模板函数）log_debug

##### 记录信息日志（模板函数）log_info

##### 记录错误日志（模板函数）log_error

## UftStrategyMgr.h/cpp — 策略管理器

### UFT策略包装器类 UftStraWrapper
```cpp
class UftStraWrapper
```

#### 成员
- **策略实例与工厂指针**
  - `UftStrategy* _stra`：策略实例指针
  - `IUftStrategyFact* _fact`：策略工厂指针

#### 获取策略实例指针 self
```cpp
/**
 * @brief 获取策略实例指针
 * @return 策略实例指针
 * 返回包装的策略实例指针。
 */
UftStrategy* self(){ return _stra; }
```

### UFT策略管理器类 UftStrategyMgr
```cpp
class UftStrategyMgr : private boost::noncopyable
```

#### 成员
- **策略工厂映射表**
  - `StraFactMap _factories`：策略工厂映射表
    - typedef wt_hashmap<std::string, `StraFactInfo`> StraFactMap; 包含
      - key为工厂名称（std::string），value为StraFactInfo结构体
      ```cpp
      /**
       * @struct StraFactInfo
      * @brief 策略工厂信息结构体
      * 存储策略工厂的相关信息，包括模块路径、句柄、工厂实例和函数指针。
      */
      typedef struct _StraFactInfo
      {
        std::string _module_path; // 模块文件路径（DLL/SO文件路径）
        DllHandle _module_inst; // 动态库句柄
        IUftStrategyFact* _fact; // 策略工厂实例指针
        FuncCreateUftStraFact _creator; // 创建工厂函数指针
        FuncDeleteUftStraFact _remover; // 删除工厂函数指针
      } StraFactInfo;
      ```

- **策略实例映射表**
  - `StrategyMap _strategies`：策略实例映射表
    - typedef wt_hashmap<std::string, `UftStrategyPtr`> StrategyMap; 包含
      - key为策略ID（std::string）
      - value为策略智能指针（UftStrategyPtr，即std::shared_ptr<UftStraWrapper>）

#### 加载策略工厂 loadFactories

#### 策略实例管理

##### 创建策略实例（完整版本）createStrategy
通过指定的工厂名称和策略名称创建策略实例：

* **查找工厂**
  * 根据传入的 `factname`（工厂名称）在 `_factories` 映射表中查找对应的工厂信息。
  * 如果找不到工厂，直接返回空指针。
* **创建与包装**
  * 从工厂信息中获取工厂接口指针 `_fact`。
  * 调用工厂接口的 `createStrategy(unitname, id)` 方法创建原始的策略对象。
  * **生命周期管理**：将创建出的原始策略指针和工厂指针一起封装进 `UftStraWrapper`，并进一步用智能指针 `UftStrategyPtr` 包装。
    * 这样做是为了确保策略删除时能正确调用工厂的删除方法，防止内存泄漏。
* **注册与返回**
  * 将生成的智能指针存入 `_strategies` 映射表，Key 为策略 ID (`id`)，以便后续查找和管理。
  * 返回该智能指针。
```cpp
/**
 * @brief 创建策略实例实现（完整版本）
 * @param factname 工厂名称
 * @param unitname 策略名称
 * @param id 策略ID
 * @return 策略智能指针，失败返回空指针
 */
UftStrategyPtr UftStrategyMgr::createStrategy(const char* factname, const char* unitname, const char* id)
```

##### 创建策略实例（简化版本）createStrategy
通过组合名称创建策略实例：
* **名称解析**
  * 调用 `StrUtil::split` 工具函数，以 `.` 为分隔符将传入的 `name` 切分。
  * **格式校验**：检查切分后的数组大小。如果小于 2（即没有找到分隔符或缺少部分），说明格式错误，直接返回空指针。
* **提取参数**
  * 将切分后的第一部分作为 `factname`（工厂名）。
  * 将切分后的第二部分作为 `unitname`（策略名）。
* **执行创建（逻辑同上一个 createStrategy）**
```cpp
/**
 * @brief 创建策略实例实现（简化版本）
 * @param name 策略名称，格式为"工厂名.策略名"
 * @param id 策略ID
 * @return 策略智能指针，失败返回空指针
 */
UftStrategyPtr UftStrategyMgr::createStrategy(const char* name, const char* id)
```

##### 获取策略实例 getStrategy
```cpp
/**
 * @brief 获取策略实例实现
 * @param id 策略ID
 * @return 策略智能指针，不存在返回空指针
 * 根据策略ID查找并返回策略实例。
 */
UftStrategyPtr UftStrategyMgr::getStrategy(const char* id)
{
	auto it = _strategies.find(id);
	if (it == _strategies.end())
		return UftStrategyPtr();
	return it->second;
}
```

# 数据管理器 WtUftDtMgr.h/cpp

## 框架图

```mermaid
graph LR
    %% 样式定义
    classDef interface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef manager fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef cache fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef notify fill:#fce4ec,stroke:#c2185b,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 接口层
    %% =======================
    subgraph InterfaceLayer["接口层"]
        direction TB
        IDataManager["IDataManager<br/>数据管理器接口<br/>定义数据查询接口"]:::interface
    end

    %% =======================
    %% 数据管理器层 - WtUftDtMgr.h/cpp
    %% =======================
    subgraph ManagerLayer["数据管理器层 - WtUftDtMgr"]
        direction TB
        
        subgraph ManagerClass["管理器类"]
            WtUftDtMgr["WtUftDtMgr<br/>UFT数据管理器<br/>实现IDataManager接口<br/>• 实时行情处理<br/>• 数据缓存管理<br/>• 数据查询接口<br/>• 数据订阅管理"]:::manager
        end
        
        subgraph CacheLayer["数据缓存层"]
            RtTickMap["_rt_tick_map<br/>实时Tick缓存<br/>存储最新Tick数据"]:::cache
            TicksCache["_ticks_cache<br/>历史Tick缓存<br/>存储历史Tick数据"]:::cache
            BarsCache["_bars_cache<br/>K线缓存<br/>存储K线数据"]:::cache
        end
        
        subgraph Subscription["订阅管理"]
            SubedBasicBars["_subed_basic_bars<br/>已订阅基础K线集合"]:::cache
        end
        
        subgraph NotifyStruct["通知结构"]
            NotifyItem["NotifyItem<br/>K线通知项"]:::notify
            BarNotifies["_bar_notifies<br/>K线通知项列表"]:::notify
        end
    end

    %% =======================
    %% 外部依赖层
    %% =======================
    subgraph ExternalLayer["外部依赖层"]
        direction TB
        WtUftEngine["WtUftEngine<br/>UFT引擎<br/>数据管理器使用者"]:::external
        WTSDataFactory["WTSDataFactory<br/>数据工厂<br/>创建数据对象"]:::external
        WTSVariant["WTSVariant<br/>配置变体类<br/>配置参数"]:::external
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    WtUftDtMgr -.->|"实现"| IDataManager

    %% =======================
    %% 组合关系（实线）
    %% =======================
    WtUftDtMgr -->|"管理"| RtTickMap
    WtUftDtMgr -->|"管理"| TicksCache
    WtUftDtMgr -->|"管理"| BarsCache
    WtUftDtMgr -->|"管理"| SubedBasicBars
    WtUftDtMgr -->|"管理"| BarNotifies
    BarNotifies -->|"包含"| NotifyItem

    %% =======================
    %% 数据流关系
    %% =======================
    WtUftEngine -->|"使用"| WtUftDtMgr
    WtUftEngine -->|"推送行情"| WtUftDtMgr
    WTSDataFactory -->|"创建数据"| WtUftDtMgr
    WTSVariant -->|"配置"| WtUftDtMgr

    %% 应用样式
    class IDataManager interface
    class WtUftDtMgr manager
    class RtTickMap,TicksCache,BarsCache,SubedBasicBars cache
    class WtUftEngine,WTSDataFactory,WTSVariant external
    class NotifyItem,BarNotifies notify
```

## UFT数据管理器类 WtUftDtMgr
```cpp
class WtUftDtMgr : public IDataManager
```

### 成员

- **核心引擎指针**
  - `WtUftEngine* _engine`：UFT引擎指针

- **数据订阅管理**
  - `wt_hashset<std::string> _subed_basic_bars`：已订阅的基础K线集合

- **数据缓存映射表**
  - `DataCacheMap* _bars_cache`：K线缓存映射表
    - `typedef WTSHashMap<std::string> DataCacheMap`：数据缓存映射表类型
  - `DataCacheMap* _ticks_cache`：历史Tick缓存映射表
    - `typedef WTSHashMap<std::string> DataCacheMap`：数据缓存映射表类型
  - `DataCacheMap* _rt_tick_map`：实时tick缓存映射表
    - `typedef WTSHashMap<std::string> DataCacheMap`：数据缓存映射表类型

- **K线通知管理**
  - `std::vector<NotifyItem> _bar_notifies`：K线通知项列表
    ```cpp
    /* @brief K线通知项结构体
    * 用于存储K线更新通知信息。*/
    typedef struct _NotifyItem
    {
        std::string _code; // 合约代码
        std::string _period; // 周期字符串
        uint32_t _times; // 周期倍数
        WTSBarStruct* _newBar; // 新的K线数据指针
    } NotifyItem;
    ```

### 初始化 init

### 处理行情推送 handle_push_quote

### IDataManager接口

#### 获取Tick数据切片 get_tick_slice

#### 获取订单队列数据切片 get_order_queue_slice

#### 获取订单明细数据切片 get_order_detail_slice

#### 获取成交明细数据切片 get_transaction_slice

#### 获取K线数据切片 get_kline_slice

#### 获取最新Tick数据 grab_last_tick

# 引擎层

## 框架图

```mermaid
graph LR
    %% 样式定义
    classDef interface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef engine fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef ticker fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef subscription fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef context fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef time fill:#e0f2f1,stroke:#004d40,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 接口层
    %% =======================
    subgraph InterfaceLayer["接口层"]
        direction TB
        IParserStub["IParserStub<br/>行情解析器存根接口<br/>定义数据推送接口"]:::interface
    end

    %% =======================
    %% 引擎层 - WtUftEngine.h/cpp
    %% =======================
    subgraph EngineLayer["引擎层 - WtUftEngine"]
        direction TB
        
        subgraph EngineClass["引擎类"]
            WtUftEngine["WtUftEngine<br/>UFT引擎<br/>实现IParserStub接口<br/>• 策略上下文管理<br/>• 数据订阅管理<br/>• 数据分发<br/>• 时间管理<br/>• 交易日管理"]:::engine
        end
        
        subgraph SubscriptionLayer["订阅管理层"]
            TickSubMap["_tick_sub_map<br/>Tick订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            OrdQueSubMap["_ordque_sub_map<br/>订单队列订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            OrdDtlSubMap["_orddtl_sub_map<br/>订单明细订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            TransSubMap["_trans_sub_map<br/>成交明细订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            BarSubMap["_bar_sub_map<br/>K线订阅映射表<br/>合约代码-周期-倍数 → 策略上下文ID集合"]:::subscription
        end
        
        subgraph ContextLayer["上下文管理层"]
            ContextMap["_ctx_map<br/>策略上下文映射表<br/>策略上下文ID → 策略上下文指针"]:::context
        end
        
        subgraph TimeLayer["时间管理层"]
            CurDate["_cur_date<br/>当前日期<br/>YYYYMMDD格式"]:::time
            CurTime["_cur_time<br/>当前时间<br/>HHMMSS格式"]:::time
            CurRawTime["_cur_raw_time<br/>原始时间<br/>HHMMSS格式"]:::time
            CurSecs["_cur_secs<br/>当前秒数<br/>包含毫秒"]:::time
            CurTDate["_cur_tdate<br/>当前交易日<br/>YYYYMMDD格式"]:::time
        end
    end

    %% =======================
    %% Ticker层 - WtUftTicker.h/cpp
    %% =======================
    subgraph TickerLayer["Ticker层 - WtUftTicker"]
        direction TB
        
        subgraph TickerClass["Ticker类"]
            WtUftRtTicker["WtUftRtTicker<br/>实时Ticker<br/>• 实时行情处理<br/>• 分钟线闭合判断<br/>• 交易日判断<br/>• 后台定时检查"]:::ticker
        end
        
        subgraph TickerTimeMgr["Ticker时间管理"]
            TickerDate["_date<br/>当前日期"]:::time
            TickerTimeVal["_time<br/>当前时间"]:::time
            CurPos["_cur_pos<br/>当前分钟位置<br/>交易时段内的分钟数"]:::time
            NextCheckTime["_next_check_time<br/>下次检查时间<br/>时间戳（毫秒）"]:::time
            LastEmitPos["_last_emit_pos<br/>上次触发分钟位置"]:::time
        end
        
        subgraph TickerThreadMgr["线程管理"]
            Stopped["_stopped<br/>停止标志"]:::time
            Thread["_thrd<br/>后台线程指针"]:::time
            Mutex["_mtx<br/>互斥锁<br/>保护共享数据"]:::time
        end
    end

    %% =======================
    %% 外部依赖层
    %% =======================
    subgraph ExternalLayer["外部依赖层"]
        direction TB
        WtUftDtMgr["WtUftDtMgr<br/>数据管理器<br/>市场数据管理"]:::external
        TraderAdapterMgr["TraderAdapterMgr<br/>交易适配器管理器<br/>交易接口管理"]:::external
        EventNotifier["EventNotifier<br/>事件通知器<br/>事件广播"]:::external
        IBaseDataMgr["IBaseDataMgr<br/>基础数据管理器<br/>合约/商品信息查询"]:::external
        WTSSessionInfo["WTSSessionInfo<br/>交易时段信息<br/>交易时间模板"]:::external
        UftStraContext["UftStraContext<br/>策略上下文<br/>策略运行环境"]:::external
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    WtUftEngine -.->|"实现"| IParserStub

    %% =======================
    %% 组合关系（实线）
    %% =======================
    WtUftEngine -->|"包含"| WtUftRtTicker
    WtUftEngine -->|"管理"| TickSubMap
    WtUftEngine -->|"管理"| OrdQueSubMap
    WtUftEngine -->|"管理"| OrdDtlSubMap
    WtUftEngine -->|"管理"| TransSubMap
    WtUftEngine -->|"管理"| BarSubMap
    WtUftEngine -->|"管理"| ContextMap
    WtUftEngine -->|"管理"| CurDate
    WtUftEngine -->|"管理"| CurTime
    WtUftEngine -->|"管理"| CurRawTime
    WtUftEngine -->|"管理"| CurSecs
    WtUftEngine -->|"管理"| CurTDate
    
    WtUftRtTicker -->|"持有"| WtUftEngine
    WtUftRtTicker -->|"管理"| TickerDate
    WtUftRtTicker -->|"管理"| TickerTimeVal
    WtUftRtTicker -->|"管理"| CurPos
    WtUftRtTicker -->|"管理"| NextCheckTime
    WtUftRtTicker -->|"管理"| LastEmitPos
    WtUftRtTicker -->|"管理"| Stopped
    WtUftRtTicker -->|"管理"| Thread
    WtUftRtTicker -->|"管理"| Mutex
    WtUftRtTicker -->|"使用"| WTSSessionInfo

    %% =======================
    %% 数据流关系
    %% =======================
    IParserStub -->|"数据推送"| WtUftEngine
    WtUftEngine -->|"转发Tick"| WtUftRtTicker
    WtUftRtTicker -->|"触发分钟闭合"| WtUftEngine
    WtUftRtTicker -->|"触发交易日事件"| WtUftEngine
    WtUftEngine -->|"分发数据"| TickSubMap
    WtUftEngine -->|"分发数据"| OrdQueSubMap
    WtUftEngine -->|"分发数据"| OrdDtlSubMap
    WtUftEngine -->|"分发数据"| TransSubMap
    WtUftEngine -->|"分发数据"| BarSubMap
    TickSubMap -->|"订阅关系"| ContextMap
    OrdQueSubMap -->|"订阅关系"| ContextMap
    OrdDtlSubMap -->|"订阅关系"| ContextMap
    TransSubMap -->|"订阅关系"| ContextMap
    BarSubMap -->|"订阅关系"| ContextMap
    ContextMap -->|"管理"| UftStraContext
    
    WtUftEngine -->|"使用"| WtUftDtMgr
    WtUftEngine -->|"使用"| TraderAdapterMgr
    WtUftEngine -->|"使用"| EventNotifier
    WtUftEngine -->|"使用"| IBaseDataMgr
    WtUftRtTicker -->|"查询"| IBaseDataMgr

    %% 应用样式
    class IParserStub interface
    class WtUftEngine engine
    class WtUftRtTicker ticker
    class TickSubMap,OrdQueSubMap,OrdDtlSubMap,TransSubMap,BarSubMap subscription
    class ContextMap context
    class CurDate,CurTime,CurRawTime,CurSecs,CurTDate,TickerDate,TickerTimeVal,CurPos,NextCheckTime,LastEmitPos,Stopped,Thread,Mutex time
    class WtUftDtMgr,TraderAdapterMgr,EventNotifier,IBaseDataMgr,WTSSessionInfo,UftStraContext external
```

## WtUftTicker.h/cpp — UFT实时ticker
```cpp
class WtUftRtTicker
```

### 成员
- **核心依赖指针**
  - `WTSSessionInfo* _s_info`：交易时段信息指针
  - `WtUftEngine* _engine`：UFT引擎指针
- **时间管理**
  - `uint32_t _date`：当前日期（YYYYMMDD格式）
  - `uint32_t _time`：当前时间（HHMMSS格式）
  - `uint32_t _cur_pos`：当前分钟位置（交易时段内的分钟数）
- **线程同步与状态**
  - `StdUniqueMutex _mtx`：互斥锁，用于保护共享数据
  - `std::atomic<uint64_t> _next_check_time`：下次检查时间（时间戳，毫秒）
  - `std::atomic<uint32_t> _last_emit_pos`：上次触发的分钟位置
  - `bool _stopped`：停止标志
  - `StdThreadPtr _thrd`：后台线程指针

### 初始化与生命周期管理

#### 初始化ticker init

#### 启动ticker run

#### 停止ticker stop

### 处理Tick数据 on_tick

## WtUftEngine.h/cpp — UFT引擎
```cpp
class WtUftEngine : public IParserStub
```

### 成员
- **时间管理**
  - `uint32_t _cur_date`：当前日期（YYYYMMDD格式）
  - `uint32_t _cur_time`：当前时间（HHMMSS格式），是1分钟线时间，比如0900，这个时候的1分钟线是0901，_cur_time也就是0901，这个是为了CTA里面方便
  - `uint32_t _cur_raw_time`：当前真实时间（HHMMSS格式）
  - `uint32_t _cur_secs`：当前秒数（包含毫秒）
  - `uint32_t _cur_tdate`：当前交易日（YYYYMMDD格式）
- **核心管理器指针**
  - `IBaseDataMgr* _base_data_mgr`：基础数据管理器指针
  - `WtUftDtMgr* _data_mgr`：数据管理器指针
  - `TraderAdapterMgr* _adapter_mgr`：交易适配器管理器指针
  - `EventNotifier* _notifier`：事件通知器指针
- **数据订阅映射表**
  - `StraSubMap _tick_sub_map`：tick数据订阅表
    - `typedef wt_hashmap<std::string, SubList> StraSubMap`：策略订阅映射表类型
    - `typedef wt_hashset<uint32_t> SubList`：订阅列表类型，策略上下文ID集合
    - 键为合约代码，值为订阅该合约的策略上下文ID集合
  - `StraSubMap _ordque_sub_map`：委托队列订阅表
    - `typedef wt_hashmap<std::string, SubList> StraSubMap`：策略订阅映射表类型
    - `typedef wt_hashset<uint32_t> SubList`：订阅列表类型，策略上下文ID集合
    - 键为合约代码，值为订阅该合约的策略上下文ID集合
  - `StraSubMap _orddtl_sub_map`：委托明细订阅表
    - `typedef wt_hashmap<std::string, SubList> StraSubMap`：策略订阅映射表类型
    - `typedef wt_hashset<uint32_t> SubList`：订阅列表类型，策略上下文ID集合
    - 键为合约代码，值为订阅该合约的策略上下文ID集合
  - `StraSubMap _trans_sub_map`：成交明细订阅表
    - `typedef wt_hashmap<std::string, SubList> StraSubMap`：策略订阅映射表类型
    - `typedef wt_hashset<uint32_t> SubList`：订阅列表类型，策略上下文ID集合
    - 键为合约代码，值为订阅该合约的策略上下文ID集合
  - `StraSubMap _bar_sub_map`：K线数据订阅表（key格式：合约代码-周期-倍数）
    - `typedef wt_hashmap<std::string, SubList> StraSubMap`：策略订阅映射表类型
    - `typedef wt_hashset<uint32_t> SubList`：订阅列表类型，策略上下文ID集合
    - 键为合约代码-周期-倍数，值为订阅该K线的策略上下文ID集合
- **策略上下文管理**
  - `ContextMap _ctx_map`：策略上下文映射表
    - `typedef wt_hashmap<uint32_t, UftContextPtr> ContextMap`：策略上下文映射表类型
    - `typedef std::shared_ptr<IUftStraCtx> UftContextPtr`：UFT策略上下文智能指针类型
    - 键为策略上下文ID，值为策略上下文智能指针
- **实时Ticker与配置**
  - `WtUftRtTicker* _tm_ticker`：实时ticker指针
  - `WTSVariant* _cfg`：配置对象指针
  - `bool _dependent`：子策略独立记账标志

### 时间管理

#### 设置日期时间 set_date_time

#### 设置交易日 set_trading_date

#### 获取当前日期 get_date

#### 获取当前分钟时间 get_min_time

#### 获取原始时间 get_raw_time

#### 获取当前秒数 get_secs

#### 获取交易日 get_trading_date

### 基础数据查询

#### 获取基础数据管理器 get_basedata_mgr

#### 获取交易时段信息 get_session_info

#### 获取商品信息 get_commodity_info

#### 获取合约信息 get_contract_info

### 数据查询接口

#### 获取最新Tick数据 get_last_tick

#### 获取Tick数据切片 get_tick_slice

#### 获取K线数据切片 get_kline_slice

#### 获取订单队列数据切片 get_order_queue_slice

#### 获取订单明细数据切片 get_order_detail_slice

#### 获取成交明细数据切片 get_transaction_slice

### 数据订阅

#### 订阅Tick数据 sub_tick

#### 订阅订单队列数据 sub_order_queue

#### 订阅订单明细数据 sub_order_detail

#### 订阅成交明细数据 sub_transaction

### 初始化与生命周期管理

#### 设置交易适配器管理器 set_adapter_mgr

#### 初始化引擎 init

#### 运行引擎 run

### 引擎生命周期与事件回调

#### 初始化完成回调 on_init

#### 交易日开始回调 on_session_begin

#### 交易日结束回调 on_session_end

#### 分钟结束回调 on_minute_end

### 数据分发

#### 处理Tick数据 on_tick

#### 处理K线数据 on_bar

### IParserStub接口实现

#### 处理行情推送 handle_push_quote

#### 处理订单明细推送 handle_push_order_detail

#### 处理订单队列推送 handle_push_order_queue

#### 处理成交明细推送 handle_push_transaction

### 策略上下文管理

#### 添加策略上下文 addContext

#### 获取策略上下文 getContext

### 通知参数更新 notify_params_update

#### 获取当前价格 get_cur_price

#### 通知参数更新 notify_params_update

# 接口层

## ITrdNotifySink.h/cpp — 交易通知接口
```cpp
class ITrdNotifySink
```

### 成交回报回调 on_trade

### 订单回报回调 on_order

### 持仓更新回调 on_position

### 交易通道就绪回调 on_channel_ready

### 交易通道丢失回调 on_channel_lost

### 下单回报回调 on_entrust

# 工具支持层

## ActionPolicyMgr.h/cpp — 动作策略管理器类
```cpp
class ActionPolicyMgr
```

### 成员
- **规则映射表**
  - `RulesMap _rules`：规则表，存储所有规则组及其规则列表
    - `typedef wt_hashmap<std::string, ActionRuleGroup> RulesMap`：规则映射表类型
    - 键为规则组名称（`std::string`），值为动作规则组（`ActionRuleGroup`）
    - `typedef std::vector<ActionRule> ActionRuleGroup`：动作规则组类型，动作规则向量
      ```cpp
      /**
       * @struct ActionRule
      * @brief 动作规则结构体
      * 定义单个动作规则的详细信息，包括动作类型、手数限制等。
      */
      typedef struct _ActionRule
      {
        ActionType _atype; // 动作类型（开仓、平仓、平今、平昨）
        uint32_t _limit; // 总手数限制（多头+空头）
        uint32_t _limit_l; // 多头手数限制
        uint32_t _limit_s; // 空头手数限制
        bool _pure; // 是否纯仓标志，主要针对AT_CloseToday和AT_CloseYestoday，用于判断是否是净今仓或者净昨仓（true表示净仓，false表示允许双向持仓）
      } ActionRule;
      ```

- **品种规则映射表**
  - `wt_hashmap<std::string, std::string> _comm_rule_map`：品种规则映射表
    - 键为合约品种代码（`std::string`），值为规则组名称（`std::string`）
    - 用于将合约品种映射到对应的规则组

### 初始化动作策略管理器 init

### 获取动作规则组 getActionRules

## EventNotifier.h/cpp — 事件通知器类
```cpp
class EventNotifier
```

### 成员
- **消息队列配置**
  - `std::string _url`：消息队列URL地址
  - `uint32_t _mq_sid`：消息队列服务器ID

- **消息队列函数指针**
  - `FuncCreateMQServer _creator`：创建消息队列服务器函数指针
    - `typedef unsigned long(*FuncCreateMQServer)(const char*)`：创建消息队列服务器函数指针类型
  - `FuncDestroyMQServer _remover`：销毁消息队列服务器函数指针
    - `typedef void(*FuncDestroyMQServer)(unsigned long)`：销毁消息队列服务器函数指针类型
  - `FundPublishMessage _publisher`：发布消息函数指针
    - `typedef void(*FundPublishMessage)(unsigned long, const char*, const char*, unsigned long)`：发布消息函数指针类型
  - `FuncRegCallbacks _register`：注册回调函数指针
    - `typedef void(*FuncRegCallbacks)(FuncLogCallback)`：注册回调函数指针类型

- **异步处理**
  - `bool _stopped`：是否已停止标志
  - `boost::asio::io_service _asyncio`：Boost异步IO服务，用于异步事件处理
  - `StdThreadPtr _worker`：异步处理工作线程指针，用于后台处理事件队列

### 初始化 init

### 事件通知接口

#### 通知成交事件 notify

#### 通知订单事件 notify

#### 通知交易消息 notify

#### 通知日志事件 notify_log

#### 通知通用事件 notify_event

### 内部辅助方法

#### 将成交信息转换为JSON格式 tradeToJson

#### 将订单信息转换为JSON格式 orderToJson

## ShareManager.h/cpp — 共享内存管理器类
```cpp
class ShareManager
```

### 成员
- **初始化与状态**
  - `bool _inited`：是否已初始化标志
  - `bool _stopped`：是否已停止标志

- **共享内存域名称**
  - `std::string _exchg`：交换区名称
  - `std::string _sync`：同步区名称

- **参数监控**
  - `wt_hashmap<std::string, uint64_t> _secnames`：监控分区名称映射表
    - 键为分区名称（`std::string`），值为最后更新时间（`uint64_t`，微秒时间戳）
  - `StdThreadPtr _worker`：监控工作线程指针，用于后台监控参数变更
  - `WtUftEngine* _engine`：UFT引擎指针，用于参数变更通知

- **动态库管理**
  - `DllHandle _inst`：动态库句柄，用于管理WtShareHelper模块
  - `std::string _module`：模块路径，存储WtShareHelper模块的完整路径

- **初始化与域管理函数指针**
  - `func_init_master _init_master`：初始化主域函数指针
    - `typedef bool (*func_init_master)(const char*, const char*)`：初始化主域函数指针类型
  - `func_get_section_updatetime _get_section_updatetime`：获取分区更新时间函数指针
    - `typedef uint64_t(*func_get_section_updatetime)(const char*, const char*)`：获取分区更新时间函数指针类型
  - `func_commit_section _commit_section`：提交分区函数指针
    - `typedef bool(*func_commit_section)(const char*, const char*)`：提交分区函数指针类型

- **设置参数函数指针**
  - `func_set_string _set_string`：设置字符串类型参数函数指针
    - `typedef bool (*func_set_string)(const char*, const char*, const char*, const char*)`：设置字符串类型参数函数指针类型
  - `func_set_int32 _set_int32`：设置int32类型参数函数指针
    - `typedef bool (*func_set_int32)(const char*, const char*, const char*, int32_t)`：设置int32类型参数函数指针类型
  - `func_set_int64 _set_int64`：设置int64类型参数函数指针
    - `typedef bool (*func_set_int64)(const char*, const char*, const char*, int64_t)`：设置int64类型参数函数指针类型
  - `func_set_uint32 _set_uint32`：设置uint32类型参数函数指针
    - `typedef bool (*func_set_uint32)(const char*, const char*, const char*, uint32_t)`：设置uint32类型参数函数指针类型
  - `func_set_uint64 _set_uint64`：设置uint64类型参数函数指针
    - `typedef bool(*func_set_uint64)(const char*, const char*, const char*, uint64_t)`：设置uint64类型参数函数指针类型
  - `func_set_double _set_double`：设置double类型参数函数指针
    - `typedef bool(*func_set_double)(const char*, const char*, const char*, double)`：设置double类型参数函数指针类型

- **获取参数函数指针**
  - `func_get_string _get_string`：获取字符串类型参数函数指针
    - `typedef const char* (*func_get_string)(const char*, const char*, const char*, const char*)`：获取字符串类型参数函数指针类型
  - `func_get_int32 _get_int32`：获取int32类型参数函数指针
    - `typedef int32_t (*func_get_int32)(const char*, const char*, const char*, int32_t)`：获取int32类型参数函数指针类型
  - `func_get_int64 _get_int64`：获取int64类型参数函数指针
    - `typedef int64_t (*func_get_int64)(const char*, const char*, const char*, int64_t)`：获取int64类型参数函数指针类型
  - `func_get_uint32 _get_uint32`：获取uint32类型参数函数指针
    - `typedef uint32_t (*func_get_uint32)(const char*, const char*, const char*, uint32_t)`：获取uint32类型参数函数指针类型
  - `func_get_uint64 _get_uint64`：获取uint64类型参数函数指针
    - `typedef uint64_t (*func_get_uint64)(const char*, const char*, const char*, uint64_t)`：获取uint64类型参数函数指针类型
  - `func_get_double _get_double`：获取double类型参数函数指针
    - `typedef double (*func_get_double)(const char*, const char*, const char*, double)`：获取double类型参数函数指针类型

- **分配参数函数指针**
  - `func_allocate_string _allocate_string`：分配字符串类型参数函数指针
    - `typedef const char*(*func_allocate_string)(const char*, const char*, const char*, const char*, bool)`：分配字符串类型参数函数指针类型
  - `func_allocate_int32 _allocate_int32`：分配int32类型参数函数指针
    - `typedef int32_t* (*func_allocate_int32)(const char*, const char*, const char*, int32_t, bool)`：分配int32类型参数函数指针类型
  - `func_allocate_int64 _allocate_int64`：分配int64类型参数函数指针
    - `typedef int64_t* (*func_allocate_int64)(const char*, const char*, const char*, int64_t, bool)`：分配int64类型参数函数指针类型
  - `func_allocate_uint32 _allocate_uint32`：分配uint32类型参数函数指针
    - `typedef uint32_t* (*func_allocate_uint32)(const char*, const char*, const char*, uint32_t, bool)`：分配uint32类型参数函数指针类型
  - `func_allocate_uint64 _allocate_uint64`：分配uint64类型参数函数指针
    - `typedef uint64_t* (*func_allocate_uint64)(const char*, const char*, const char*, uint64_t, bool)`：分配uint64类型参数函数指针类型
  - `func_allocate_double _allocate_double`：分配double类型参数函数指针
    - `typedef double*	(*func_allocate_double)(const char*, const char*, const char*, double, bool)`：分配double类型参数函数指针类型

### 单例模式与核心属性

#### 获取单例实例 self

#### 设置UFT引擎指针 set_engine

### 初始化与生命周期管理

#### 初始化共享内存管理器 initialize

#### 启动参数监控 start_watching

#### 初始化共享内存域 init_domain

#### 提交参数监控分区 commit_param_watcher

### 参数设置接口

#### 设置字符串类型参数 set_value

#### 设置int32类型参数 set_value

#### 设置int64类型参数 set_value

#### 设置uint32类型参数 set_value

#### 设置uint64类型参数 set_value

#### 设置double类型参数 set_value

### 参数获取接口

#### 获取字符串类型参数 get_value

#### 获取int32类型参数 get_value

#### 获取int64类型参数 get_value

#### 获取uint32类型参数 get_value

#### 获取uint64类型参数 get_value

#### 获取double类型参数 get_value

### 参数分配接口（返回指针，支持直接修改）

#### 分配字符串类型字段 allocate_value

#### 分配int32类型字段 allocate_value

#### 分配int64类型字段 allocate_value

#### 分配uint32类型字段 allocate_value

#### 分配uint64类型字段 allocate_value

#### 分配double类型字段 allocate_value

## WtHelper.h/cpp — 辅助工具类
```cpp
class WtHelper
```

### 成员
- **时间状态（静态成员变量）**
  - `static uint32_t _cur_date`：当前日期（YYYYMMDD格式）
  - `static uint32_t _cur_time`：当前时间（HHMMSS格式），以分钟为准
  - `static uint32_t _cur_secs`：当前秒数（包含毫秒）
  - `static uint32_t _cur_tdate`：当前交易日（YYYYMMDD格式）

- **目录路径（静态成员变量）**
  - `static std::string _inst_dir`：实例所在目录
  - `static std::string _gen_dir`：生成文件输出目录

### 路径管理

#### 获取当前工作目录 getCWD

#### 获取模块路径 getModulePath

#### 获取基础目录 getBaseDir

#### 获取输出目录 getOutputDir

#### 获取策略数据目录 getStraDataDir

#### 获取策略用户数据目录 getStraUsrDatDir

#### 获取投资组合目录 getPortifolioDir

#### 获取实例目录 getInstDir

#### 设置实例目录 setInstDir

#### 设置生成目录 setGenerateDir

### 时间管理

#### 设置当前时间 setTime

#### 设置当前交易日 setTDate

#### 获取当前日期 getDate

#### 获取当前时间 getTime

#### 获取当前秒数 getSecs

#### 获取当前交易日 getTradingDate